# AgroInsight — Análisis Completo de Producción Agrícola Colombia
## Evaluaciones Agropecuarias Municipales EVA V1 · 2006–2018
**Bootcamp Talento Tech — IA Nivel Integrador · Universidad de Antioquia**

### Estructura del notebook
| Sección | Contenido | Tiempo estimado |
|---|---|---|
| 0 | Instalación y Google Drive | 2 min |
| 1 | Carga, limpieza y EDA | 5 min |
| 2 | M1 — Regresión de producción (RF vs NN) | 10 min |
| 3 | M2 — Clasificación alta/baja (SVM vs NN) | 8 min |
| 4 | M3 — Clustering municipios + Mapas | 10 min |
| 5 | Guardar modelos en Google Drive | 2 min |
| 6 | **Cargar modelos** (usar el día de la presentación) | 1 min |
| 7 | App Gradio | 1 min |

### Para la presentación
**Primera vez:** Correr secciones 0 → 5 (entrena y guarda todo)

**Día de la presentación:** Correr secciones 0 → 1 → 6 → 7 (carga modelos y lanza app)

---

## Sección 0 — Instalación de dependencias y Google Drive

Ejecutar siempre al iniciar la sesión de Colab.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Instalación de dependencias
# pandas / numpy:   manipulación de datos
# sodapy:           cliente API Socrata (fuente del dataset EVA)
# statsmodels:      análisis VIF de multicolinealidad
# plotly:           mapas interactivos M3
# streamlit:        app de presentación local
# ─────────────────────────────────────────────────────────────
!pip install pandas sodapy statsmodels plotly streamlit -q
print('Dependencias instaladas OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─────────────────────────────────────────────────────────────
# Rutas del proyecto en Google Drive
# Estructura organizada por carpetas
# ─────────────────────────────────────────────────────────────
RUTA_DRIVE    = '/content/drive/MyDrive/AgroInsight'
RUTA_MODELOS  = f'{RUTA_DRIVE}/modelos'
RUTA_GRAFICAS = f'{RUTA_DRIVE}/graficas'
RUTA_MAPAS    = f'{RUTA_DRIVE}/mapas'

os.makedirs(RUTA_MODELOS,  exist_ok=True)
os.makedirs(RUTA_GRAFICAS, exist_ok=True)
os.makedirs(RUTA_MAPAS,    exist_ok=True)

print(f'Drive montado OK')
print(f'  Modelos:  {RUTA_MODELOS}')
print(f'  Graficas: {RUTA_GRAFICAS}')
print(f'  Mapas:    {RUTA_MAPAS}')

---
## Importaciones globales

Todas las dependencias del proyecto se importan aquí una sola vez.
Las celdas de cada módulo contienen únicamente el código de análisis.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Librerías estándar
# ─────────────────────────────────────────────────────────────
import os
import json
import shutil
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# Ciencia de datos
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# Visualización
# ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import requests

# ─────────────────────────────────────────────────────────────
# Persistencia de modelos
# ─────────────────────────────────────────────────────────────
import joblib

# ─────────────────────────────────────────────────────────────
# Scikit-learn — preprocesamiento
# ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, LabelEncoder
)
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ─────────────────────────────────────────────────────────────
# Scikit-learn — modelos y búsqueda
# ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ─────────────────────────────────────────────────────────────
# Scikit-learn — métricas
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, roc_curve
)

# ─────────────────────────────────────────────────────────────
# TensorFlow / Keras
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Embedding, Flatten, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)
print('Importaciones globales cargadas OK')
print(f'  numpy:      {np.__version__}')
print(f'  pandas:     {pd.__version__}')
print(f'  tensorflow: {tf.__version__}')

---
## Sección 1 — Carga de datos, limpieza y análisis exploratorio

Pipeline compartido para los 3 módulos — se ejecuta **una sola vez**.
Fuente: MADR — Evaluaciones Agropecuarias Municipales EVA V1 · 2006–2018 · datos.gov.co


## 1. Carga de datos desde API Socrata

In [ ]:
# Configuración de visualización inline para Colab
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size']      = 11
print('Configuración de gráficas OK')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Carga desde API Socrata (datos.gov.co)
# Dataset: Evaluaciones Agropecuarias Municipales EVA V1
# ID: 2pnw-mmge | Registros esperados: ~205,242
# ─────────────────────────────────────────────────────────────
from sodapy import Socrata

cliente = Socrata('www.datos.gov.co', None)
resultados = cliente.get('2pnw-mmge', limit=210_000)
resultados_df = pd.DataFrame.from_records(resultados)

print(f'Registros cargados: {len(resultados_df):,}')
print(f'Columnas:           {resultados_df.shape[1]}')
resultados_df.info()

*Vista previa del dataset — ver salida de celda anterior.*

## 2. Transformación de tipos de dato

Todas las columnas numéricas llegan como `object` desde la API. Se convierten con `errors='coerce'` para manejar valores no parseables de forma segura.

In [ ]:
# Conversion de columnas numericas que llegan como object desde la API
resultados_df['a_o']              = pd.to_numeric(resultados_df['a_o'],              errors='coerce').astype('Int64')
resultados_df['rea_sembrada_ha']  = pd.to_numeric(resultados_df['rea_sembrada_ha'],  errors='coerce').astype('float')
resultados_df['rea_cosechada_ha'] = pd.to_numeric(resultados_df['rea_cosechada_ha'], errors='coerce').astype('float')
resultados_df['producci_n_t']     = pd.to_numeric(resultados_df['producci_n_t'],     errors='coerce').astype('float')
resultados_df['rendimiento_t_ha'] = pd.to_numeric(resultados_df['rendimiento_t_ha'], errors='coerce').astype('float')

print('Tipos de dato despues de la conversion:')
print(resultados_df.dtypes)

## 3. Limpieza de datos

In [ ]:
data = resultados_df.copy()

# Eliminar columnas innecesarias para el analisis
cols_a_eliminar = ['c_d_dep', 'c_d_mun', 'nombre_cientifico']
data.drop(columns=cols_a_eliminar, inplace=True, errors='ignore')

print('Columnas despues de eliminar innecesarias:')
print(data.columns.tolist())

In [ ]:
# Mantener fila si AL MENOS UNA columna numerica es mayor a cero
# Solo se elimina si sembrada=0 Y cosechada=0 Y produccion=0 simultaneamente
data_limpia = data[
    (data['rea_sembrada_ha']  > 0) |
    (data['rea_cosechada_ha'] > 0) |
    (data['producci_n_t']     > 0)
].copy()

filas_eliminadas = len(data) - len(data_limpia)
print(f'Registros eliminados (todas las columnas = 0): {filas_eliminadas:,}')
print(f'Registros restantes: {len(data_limpia):,}')

In [ ]:
# Renombrar columnas a nombres limpios y consistentes
data_limpia.columns = [
    'departamento', 'municipio', 'grupo_cultivo',
    'subgrupo_cultivo', 'cultivo', 'desagregacion', 'anio', 'periodo',
    'area_sembrada_ha', 'area_cosechada_ha', 'produccion_t',
    'rendimiento_t_ha', 'estado_fisico', 'ciclo_cultivo'
]

# Verificar sobre data_limpia con nombres ya aplicados
print('Columnas renombradas exitosamente:')
print(data_limpia.columns.tolist())

## 4. Análisis de categorías

In [ ]:
# Usar data_limpia con columnas ya renombradas
print('Valores unicos por columna categorica:')
for col in data_limpia.select_dtypes(include='object').columns:
    n_unique = data_limpia[col].nunique()
    ejemplos = data_limpia[col].unique()[:5]
    print(f'  {col}: {n_unique} categorias -> {list(ejemplos)}')

## 5. Detección e imputación de datos faltantes

In [ ]:
print('=' * 60)
print('ANALISIS DE VALORES NULOS POR COLUMNA')
print('=' * 60)

nulos      = data_limpia.isnull().sum()
porcentaje = (nulos / len(data_limpia) * 100).round(2)  # denominador correcto: data_limpia

df_nulos = pd.DataFrame({
    'Columna':    data_limpia.columns,
    'Nulos':      nulos.values,
    'Porcentaje': porcentaje.values
}).sort_values('Nulos', ascending=False)

for _, row in df_nulos.iterrows():
    estado = '[NULO]' if row['Nulos'] > 0 else '[OK]'
    print(f'  {estado} {row["Columna"]:30s}: {row["Nulos"]:6.0f} ({row["Porcentaje"]:5.2f}%)')

print(f'\nTotal nulos: {nulos.sum():,}')
print(f'Filas completas: {data_limpia.dropna().shape[0]:,} de {len(data_limpia):,}')

**Estrategia de imputacion:**

1. `rendimiento_t_ha`: se imputa con **0** — en todos los casos nulos, `area_cosechada_ha` tambien es 0, lo que indica que no se logro cosechar nada de lo sembrado.
2. `municipio`: el unico faltante corresponde a **BAJO BAUDO**, identificado por su codigo DANE en la data original sin modificar.

In [ ]:
# Imputacion de faltantes
data_limpia['rendimiento_t_ha'] = data_limpia['rendimiento_t_ha'].fillna(0)
data_limpia['municipio']        = data_limpia['municipio'].fillna('BAJO BAUDO')

# Crear variable es_semestral
data_limpia['es_semestral'] = data_limpia['periodo'].apply(
    lambda x: 1 if str(x).strip().endswith('A') or str(x).strip().endswith('B') else 0
)

print(f"Nulos en 'rendimiento_t_ha': {data_limpia['rendimiento_t_ha'].isnull().sum()}")
print(f"Nulos en 'municipio':        {data_limpia['municipio'].isnull().sum()}")
print(f"\nDistribucion es_semestral:")
print(data_limpia['es_semestral'].value_counts().to_string())
print(f"  0 = Ciclo anual (frutales, permanentes)")
print(f"  1 = Ciclo semestral (cereales, hortalizas, leguminosas)")

---
## 6. Justificacion analisis de Outliers

Los datos atipicos identificados fueron conservados en el conjunto de datos, dado que su presencia refleja variabilidad real inherente al sector agropecuario colombiano. En el contexto de las Evaluaciones Agropecuarias Municipales, estos valores extremos corresponden frecuentemente a municipios con siembras de gran escala en cultivos especificos — por ejemplo, la cana azucarera en el Valle del Cauca puede superar los 4.5 millones de toneladas de produccion anual. Eliminar estos registros implicaria perdida de informacion tecnica valiosa e introduciria sesgos artificiales que distorsionarian la distribucion real de los datos.

---

### Boxplot de Produccion por Año — Escala original

Se presentan dos versiones del boxplot para evidenciar el efecto de los outliers en la visualizacion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Escala original
sns.boxplot(ax=axes[0], x='anio', y=data_limpia['produccion_t'],
            data=data_limpia, palette='pastel', hue='anio', legend=False)
axes[0].set_title('Produccion por Ano — Escala Original', fontsize=14)
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Produccion (toneladas)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Escala logaritmica
sns.boxplot(ax=axes[1], x='anio', y=np.log1p(data_limpia['produccion_t']),
            data=data_limpia, palette='pastel', hue='anio', legend=False)
axes[1].set_title('Produccion por Ano — Escala Logaritmica', fontsize=14)
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Log(Produccion)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle('Distribucion de la Produccion Agricola 2006-2018', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Interpretacion:** La escala original comprime la mayoria de distribuciones por los outliers extremos. La escala logaritmica revela una tendencia al alza en la produccion de 2006 a 2017, con variabilidad creciente en anos recientes.

## 7. Matriz de correlacion y VIF

In [ ]:
numericas    = data_limpia.select_dtypes(include='number')
corr_matrix  = numericas.corr()

print('CORRELACIONES ENTRE VARIABLES NUMERICAS:')
print(corr_matrix.round(3).to_string())

# Heatmap
plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Mapa de Calor — Correlaciones Dataset EVA Agricola', pad=15)
plt.tight_layout()
plt.show()

# Top correlaciones
print('\nTOP CORRELACIONES (excluyendo diagonal):')
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append({
            'par': f'{corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}',
            'correlacion': abs(corr_matrix.iloc[i, j])
        })
df_corr = pd.DataFrame(corr_pairs).sort_values('correlacion', ascending=False)
print(df_corr.head(10).to_string(index=False))

### Deteccion de Multicolinealidad con VIF

El VIF (Factor de Inflacion de Varianza) mide cuanto se infla la varianza de un coeficiente por multicolinealidad:
- VIF = 1: sin correlacion
- VIF 1–5: multicolinealidad moderada
- VIF > 5: multicolinealidad severa (problematica)

In [ ]:
X_vif = data_limpia[numericas.columns].drop(columns=['anio'], errors='ignore')

vif_data          = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF']     = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print('Analisis de Multicolinealidad (VIF):')
display(vif_data.sort_values(by='VIF', ascending=False))

high_vif = vif_data[vif_data['VIF'] > 5]
if not high_vif.empty:
    print('\nAlerta: variables con VIF > 5 (multicolinealidad severa):')
    print(high_vif.to_string())
    print('\nEstrategia: se aplicara PCA para el RF y Embeddings para la NN.')
else:
    print('\nNo se detecto multicolinealidad severa (VIF < 5).')

**Estrategia según modelo:**
- **Random Forest M1 y M2**: maneja multicolinealidad internamente — se entrena directamente sobre las 50 features OHE sin necesitar PCA.
- **Red Neuronal M1 y M2**: usa Embeddings para variables categóricas — aprende representaciones densas sin OHE ni PCA.
- **Ventaja OHE directo**: con variables binarias dispersas el PCA solo captura ~8% de varianza por componente — es más efectivo usar los modelos directamente sobre el espacio completo.


## 8. Preparación de features para los modelos

Se construye la matriz de features compartida usando:
- **Variables numéricas:** `area_sembrada_ha` y `es_semestral`
- **Variables categóricas (OHE):** `departamento`, `grupo_cultivo`, `ciclo_cultivo`

Esta misma estrategia se aplica en M1 y M2. Las features representan únicamente
información disponible **antes de sembrar** — sin `rendimiento_t_ha` (dato post-cosecha).


In [ ]:
# ─────────────────────────────────────────────────────────────
# Preparación de features M1 — Regresión de producción
# Features: solo información disponible ANTES de sembrar
# Target:   log1p(produccion_t) — distribución log-normal
# ─────────────────────────────────────────────────────────────
FEATURES_NUM_M1 = ['area_sembrada_ha', 'es_semestral']
FEATURES_CAT_M1 = ['departamento', 'grupo_cultivo', 'ciclo_cultivo']

df_m1 = data_limpia[FEATURES_NUM_M1 + FEATURES_CAT_M1 + ['produccion_t']].copy().reset_index(drop=True)

# One-Hot Encoding de variables categóricas
ohe_m1   = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_m1 = ohe_m1.fit_transform(df_m1[FEATURES_CAT_M1])

# Escalado de variables numéricas
scaler_num_m1 = StandardScaler()
X_num_m1      = scaler_num_m1.fit_transform(df_m1[FEATURES_NUM_M1])

# Matriz de features final: numéricas + OHE categóricas
X_m1 = np.hstack([X_num_m1, X_cat_m1])

# Target en escala logarítmica
# Justificación: produccion_t tiene distribución log-normal (0 a 4.5M t)
# El R² en escala log es la métrica honesta — la app convierte con expm1
LOG_MAX_M1 = np.log1p(df_m1['produccion_t'].max())
y_log_m1   = np.log1p(df_m1['produccion_t'].values)

# Niveles interpretativos basados en quintiles reales del dataset
NIVELES_PRODUCCION = {
    1: {'nombre': 'Subsistencia',       'rango': '< 30 t',        'limite': 30},
    2: {'nombre': 'Pequeño productor',  'rango': '30–105 t',      'limite': 105},
    3: {'nombre': 'Mediano',            'rango': '105–329 t',     'limite': 329},
    4: {'nombre': 'Grande',             'rango': '329–1,236 t',   'limite': 1236},
    5: {'nombre': 'Agroindustrial',     'rango': '> 1,236 t',     'limite': float('inf')},
}

def clasificar_nivel_produccion(toneladas):
    """Convierte toneladas predichas a nivel interpretable (1-5)."""
    for nivel, info in NIVELES_PRODUCCION.items():
        if toneladas < info['limite']:
            return nivel, f"Nivel {nivel} — {info['nombre']} ({info['rango']})"
    return 5, f"Nivel 5 — Agroindustrial (> 1,236 t)"

print(f'Features numéricas M1:   {FEATURES_NUM_M1}')
print(f'Features categóricas M1: {FEATURES_CAT_M1}')
print(f'Columnas OHE:            {X_cat_m1.shape[1]}')
print(f'Total features:          {X_m1.shape[1]}')
print(f'Target: log1p(produccion_t)')
print(f'  Mediana en toneladas:  {np.expm1(np.median(y_log_m1)):.0f} t')
print(f'  Máximo en toneladas:   {np.expm1(LOG_MAX_M1):,.0f} t')

In [ ]:
# Split 80/20 — mismo random_state en M1 y M2 para comparativa justa
X_train_m1, X_test_m1, y_train_log_m1, y_test_log_m1 = train_test_split(
    X_m1, y_log_m1, test_size=0.2, random_state=42
)

print(f'Entrenamiento M1: {X_train_m1.shape[0]:,} registros x {X_train_m1.shape[1]} features')
print(f'Prueba M1:        {X_test_m1.shape[0]:,} registros')
print(f'Target log — rango: [{y_train_log_m1.min():.2f}, {y_train_log_m1.max():.2f}]')

---
## Sección 2 — Módulo 1: Regresión de Producción Agrícola

**Objetivo:** Predecir la producción en toneladas dado un conjunto de condiciones de siembra.

**Target:** `log1p(produccion_t)` — escala logarítmica por distribución log-normal del dataset.
La app convierte con `expm1` para mostrar toneladas al usuario.

**Comparativa ML vs DL:**
| Modelo | Tipo | Estrategia |
|---|---|---|
| Random Forest | ML | Ensemble de árboles sobre OHE (50 features) |
| Red Neuronal con Embeddings | DL | Embeddings para categóricas + capas densas |


---
## 9. Módulo 1 — Comparativa ML vs DL: Regresión de Producción

**Variable objetivo:** `log1p(produccion_t)` — producción en toneladas transformada a escala log.
**Features:** `area_sembrada_ha`, `es_semestral` + OHE(`departamento`, `grupo_cultivo`, `ciclo_cultivo`)
**Métrica principal:** R² en escala log | **Referencia app:** toneladas via `expm1`


### 9.1 Random Forest Regressor

In [ ]:
# Random Forest usa las mismas features y target que el split general de M1
X_train_rf_m1, X_test_rf_m1 = X_train_m1, X_test_m1
y_train_rf_m1, y_test_rf_m1 = y_train_log_m1, y_test_log_m1

print(f'Entrenamiento RF M1: {X_train_rf_m1.shape[0]:,} × {X_train_rf_m1.shape[1]} features')
print(f'Prueba RF M1:        {X_test_rf_m1.shape[0]:,}')
print(f'Target: log1p(produccion_t) — RF robusto a alta dimensión sin PCA')

#### Modelo base — linea de referencia

In [ ]:
# Modelo base RF M1 — línea de referencia sin optimizar
print('Entrenando Random Forest M1 base (sin optimizar)...')
rf_base_m1 = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_base_m1.fit(X_train_rf_m1, y_train_rf_m1)

y_pred_rf_m1_base = rf_base_m1.predict(X_test_rf_m1)

r2_rf_m1_base   = r2_score(y_test_rf_m1,  y_pred_rf_m1_base)
rmse_rf_m1_base = np.sqrt(mean_squared_error(y_test_rf_m1, y_pred_rf_m1_base))

print(f'RF M1 Base — R² log: {r2_rf_m1_base:.4f} | RMSE log: {rmse_rf_m1_base:.4f}')

#### Optimizacion con GridSearchCV

In [ ]:
# ─────────────────────────────────────────────────────────────
# Optimización RF M1 con GridSearchCV
# Grid reducido para tiempo razonable en Colab
# 8 combinaciones × 3 folds = 24 entrenamientos (~5-8 min)
# Los parámetros más impactantes en RF sobre datasets grandes:
#   n_estimators: más árboles = más estable
#   max_depth:    controla sobreajuste
#   min_samples_split: regularización
# ─────────────────────────────────────────────────────────────
param_grid_rf_m1 = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 10],
    'min_samples_split': [2, 5],
}

grid_rf_m1 = GridSearchCV(
    estimator  = RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid = param_grid_rf_m1,
    cv         = 3,
    n_jobs     = -1,
    verbose    = 1,
    scoring    = 'neg_mean_squared_error',
)

print('Iniciando GridSearch RF M1...')
print(f'Combinaciones: {len(param_grid_rf_m1["n_estimators"]) * len(param_grid_rf_m1["max_depth"]) * len(param_grid_rf_m1["min_samples_split"])} × 3 folds = 24 entrenamientos')
grid_rf_m1.fit(X_train_rf_m1, y_train_rf_m1)
print(f'Mejores parámetros: {grid_rf_m1.best_params_}')
rf_opt_m1 = grid_rf_m1.best_estimator_

#### Evaluacion completa del RF

In [ ]:
# ─────────────────────────────────────────────────────────────
# Evaluación Random Forest M1
# Métricas en escala log (indicador académico principal)
# Conversión a toneladas con expm1 (referencia para la app)
# ─────────────────────────────────────────────────────────────

# Predicciones en escala log con clip para evitar explosión en expm1
y_pred_rf_m1_train_log = np.clip(rf_opt_m1.predict(X_train_rf_m1), 0, LOG_MAX_M1)
y_pred_rf_m1_test_log  = np.clip(rf_opt_m1.predict(X_test_rf_m1),  0, LOG_MAX_M1)

# Métricas en escala log — métrica principal
r2_rf_m1_train    = r2_score(y_train_rf_m1, y_pred_rf_m1_train_log)
r2_rf_m1_test     = r2_score(y_test_rf_m1,  y_pred_rf_m1_test_log)
rmse_rf_m1_train  = np.sqrt(mean_squared_error(y_train_rf_m1, y_pred_rf_m1_train_log))
rmse_rf_m1_test   = np.sqrt(mean_squared_error(y_test_rf_m1,  y_pred_rf_m1_test_log))
mae_rf_m1_test    = mean_absolute_error(y_test_rf_m1, y_pred_rf_m1_test_log)
gap_rf_m1         = r2_rf_m1_train - r2_rf_m1_test

# Convertir a toneladas para referencia en app
y_pred_rf_m1_test_ton  = np.expm1(y_pred_rf_m1_test_log)
y_pred_rf_m1_train_ton = np.expm1(y_pred_rf_m1_train_log)
y_test_rf_m1_ton       = np.expm1(y_test_rf_m1)
y_train_rf_m1_ton      = np.expm1(y_train_rf_m1)

r2_rf_m1_test_ton   = r2_score(y_test_rf_m1_ton, y_pred_rf_m1_test_ton)
rmse_rf_m1_test_ton = np.sqrt(mean_squared_error(y_test_rf_m1_ton, y_pred_rf_m1_test_ton))

print('=== Random Forest M1 ===')
print(f'Métrica principal — escala log:')
print(f'  R² entrenamiento: {r2_rf_m1_train:.4f}')
print(f'  R² prueba:        {r2_rf_m1_test:.4f}')
print(f'  RMSE log prueba:  {rmse_rf_m1_test:.4f}')
print(f'  Gap train/test:   {gap_rf_m1:.4f}{"  ← overfitting" if gap_rf_m1 > 0.05 else "  ← generaliza bien"}')
print(f'Referencia app — toneladas:')
print(f'  R² prueba (t):    {r2_rf_m1_test_ton:.4f}')
print(f'  RMSE prueba (t):  {rmse_rf_m1_test_ton:,.0f} t')

#### Visualizacion del desempeno del RF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Actual vs Predicho (en toneladas — más interpretable)
axes[0].scatter(y_test_rf_m1_ton, y_pred_rf_m1_test_ton, alpha=0.3, color='steelblue', s=10)
lim = max(y_test_rf_m1_ton.max(), y_pred_rf_m1_test_ton.max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Producción Real (t)')
axes[0].set_ylabel('Producción Predicha (t)')
axes[0].set_title(f'RF: Actual vs Predicho (R²={r2_rf_m1_test_ton:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Residuos en escala log
residuos_rf_m1 = y_test_rf_m1 - y_pred_rf_m1_test_log
axes[1].scatter(y_pred_rf_m1_test_log, residuos_rf_m1, alpha=0.3, color='coral', s=10)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Producción Predicha (log)')
axes[1].set_ylabel('Residuos (Real − Predicho) en log')
axes[1].set_title('RF: Gráfico de Residuos (escala log)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Desempeño Random Forest M1 — Conjunto de Prueba', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 9.2 Red Neuronal MLP con Embeddings (Deep Learning)

**¿Por que Embeddings?**

Las capas de Embedding aprenden **representaciones densas** de variables categoricas. A diferencia del One-Hot Encoding que trata cada categoria como independiente y ortogonal, los Embeddings descubren similitudes entre categorias — por ejemplo, que Boyaca y Cundinamarca tienen patrones agricolas similares, o que Frutales y Permanentes comparten dinamicas de produccion. Esta capacidad de aprendizaje representacional es **exclusiva del Deep Learning** y constituye el diferencial academico central frente al Random Forest.

**Variables de entrada:**
- Numericas escaladas: `area_sembrada_ha`, `area_cosechada_ha`, `rendimiento_t_ha`, `anio`
- Categoricas con Embedding: `departamento`, `grupo_cultivo`, `ciclo_cultivo`

In [ ]:
# ─────────────────────────────────────────────────────────────
# Preparación para Red Neuronal M1
# La NN usa Embeddings para categóricas — no necesita OHE
# Cada categoría se representa como un vector denso aprendido
# ─────────────────────────────────────────────────────────────
df_nn_m1 = data_limpia.copy().reset_index(drop=True)

# LabelEncoder — convierte categorías a enteros para los Embeddings
le_dep_m1   = LabelEncoder()
le_grp_m1   = LabelEncoder()
le_ciclo_m1 = LabelEncoder()

df_nn_m1['dep_enc']   = le_dep_m1.fit_transform(df_nn_m1['departamento'])
df_nn_m1['grp_enc']   = le_grp_m1.fit_transform(df_nn_m1['grupo_cultivo'])
df_nn_m1['ciclo_enc'] = le_ciclo_m1.fit_transform(df_nn_m1['ciclo_cultivo'])

n_departamentos = df_nn_m1['dep_enc'].nunique()
n_grupos        = df_nn_m1['grp_enc'].nunique()
n_ciclos        = df_nn_m1['ciclo_enc'].nunique()

# Features numéricas — mismas que el RF para comparativa justa
# SIN rendimiento_t_ha (dato post-cosecha, no disponible antes de sembrar)
FEATURES_NUM_NN_M1 = ['area_sembrada_ha', 'es_semestral']
scaler_nn_m1       = StandardScaler()
X_num_nn_m1        = scaler_nn_m1.fit_transform(df_nn_m1[FEATURES_NUM_NN_M1])

# Target: log1p(produccion_t) — igual que el RF
y_nn_m1 = np.log1p(df_nn_m1['produccion_t'].values)

# Arrays de entrada categórica
X_dep_nn_m1   = df_nn_m1['dep_enc'].values
X_grp_nn_m1   = df_nn_m1['grp_enc'].values
X_ciclo_nn_m1 = df_nn_m1['ciclo_enc'].values

print(f'Features numéricas NN M1:  {FEATURES_NUM_NN_M1}')
print(f'Categorías para Embeddings:')
print(f'  departamento:  {n_departamentos} categorías')
print(f'  grupo_cultivo: {n_grupos} categorías')
print(f'  ciclo_cultivo: {n_ciclos} categorías')
print(f'Target: log1p(produccion_t) — misma escala que RF')

In [ ]:
# Split NN M1 — mismo random_state que RF
indices_nn_m1 = np.arange(len(df_nn_m1))
idx_train_m1, idx_test_m1 = train_test_split(
    indices_nn_m1, test_size=0.2, random_state=42
)

X_num_train_m1,   X_num_test_m1   = X_num_nn_m1[idx_train_m1],   X_num_nn_m1[idx_test_m1]
X_dep_train_m1,   X_dep_test_m1   = X_dep_nn_m1[idx_train_m1],   X_dep_nn_m1[idx_test_m1]
X_grp_train_m1,   X_grp_test_m1   = X_grp_nn_m1[idx_train_m1],   X_grp_nn_m1[idx_test_m1]
X_ciclo_train_m1, X_ciclo_test_m1 = X_ciclo_nn_m1[idx_train_m1], X_ciclo_nn_m1[idx_test_m1]
y_train_nn_m1,    y_test_nn_m1    = y_nn_m1[idx_train_m1],        y_nn_m1[idx_test_m1]

print(f'Entrenamiento NN M1: {len(idx_train_m1):,} registros')
print(f'Prueba NN M1:        {len(idx_test_m1):,} registros')
print(f'Target log rango:    [{y_train_nn_m1.min():.2f}, {y_train_nn_m1.max():.2f}]')

#### Arquitectura con Embeddings

In [ ]:
tf.random.set_seed(42)

n_num_features_m1 = X_num_train_m1.shape[1]
dim_dep_m1   = min(50, n_departamentos // 2 + 1)
dim_grp_m1   = min(50, n_grupos        // 2 + 1)
dim_ciclo_m1 = min(50, n_ciclos        // 2 + 1)

inp_num_m1   = Input(shape=(n_num_features_m1,), name='numericas')
inp_dep_m1   = Input(shape=(1,),                 name='departamento')
inp_grp_m1   = Input(shape=(1,),                 name='grupo_cultivo')
inp_ciclo_m1 = Input(shape=(1,),                 name='ciclo_cultivo')

emb_dep_m1   = Flatten()(Embedding(n_departamentos, dim_dep_m1,   name='emb_dep')(inp_dep_m1))
emb_grp_m1   = Flatten()(Embedding(n_grupos,        dim_grp_m1,   name='emb_grp')(inp_grp_m1))
emb_ciclo_m1 = Flatten()(Embedding(n_ciclos,        dim_ciclo_m1, name='emb_ciclo')(inp_ciclo_m1))

x_m1 = Concatenate()([inp_num_m1, emb_dep_m1, emb_grp_m1, emb_ciclo_m1])
x_m1 = Dense(128, activation='relu')(x_m1)
x_m1 = BatchNormalization()(x_m1)
x_m1 = Dropout(0.3)(x_m1)
x_m1 = Dense(64, activation='relu')(x_m1)
x_m1 = BatchNormalization()(x_m1)
x_m1 = Dropout(0.2)(x_m1)
x_m1 = Dense(32, activation='relu')(x_m1)

salida_m1 = Dense(1, activation='linear', name='prediccion_log')(x_m1)

model_nn_m1 = Model(
    inputs  = [inp_num_m1, inp_dep_m1, inp_grp_m1, inp_ciclo_m1],
    outputs = salida_m1,
    name    = 'AgroInsight_M1_NN'
)

model_nn_m1.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_nn_m1.summary()

#### Entrenamiento con EarlyStopping

In [ ]:
# ─────────────────────────────────────────────────────────────
# Entrenamiento NN M1 con EarlyStopping + ReduceLROnPlateau
# EarlyStopping: detiene si val_loss no mejora en 20 epochs
# ReduceLROnPlateau: reduce lr si val_loss estanca (ayuda convergencia)
# batch_size=512: optimizado para GPU de Colab
# ─────────────────────────────────────────────────────────────
early_stop_m1 = EarlyStopping(
    monitor             = 'val_loss',
    patience            = 20,
    restore_best_weights = True,
    verbose             = 1,
)

reduce_lr_m1 = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 8,
    min_lr   = 1e-6,
    verbose  = 1,
)

history_m1 = model_nn_m1.fit(
    x = {'numericas':    X_num_train_m1,
         'departamento': X_dep_train_m1.reshape(-1, 1),
         'grupo_cultivo':X_grp_train_m1.reshape(-1, 1),
         'ciclo_cultivo':X_ciclo_train_m1.reshape(-1, 1)},
    y                = y_train_nn_m1,
    validation_split = 0.15,
    epochs           = 200,
    batch_size       = 512,
    callbacks        = [early_stop_m1, reduce_lr_m1],
    verbose          = 1,
)

print(f'Epochs ejecutados: {len(history_m1.history["loss"])}')
print(f'Mejor val_loss:    {min(history_m1.history["val_loss"]):.4f}')

#### Curva de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_m1.history['loss'],     label='Entrenamiento', color='steelblue')
axes[0].plot(history_m1.history['val_loss'], label='Validacion',    color='orange')
axes[0].set_title('Curva de Perdida (MSE) por Epoca')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_m1.history['mae'],     label='Entrenamiento', color='steelblue')
axes[1].plot(history_m1.history['val_mae'], label='Validacion',    color='orange')
axes[1].set_title('Curva de MAE por Epoca')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — Red Neuronal con Embeddings', fontsize=14)
plt.tight_layout()
plt.show()

#### Evaluacion de la Red Neuronal

**Nota tecnica sobre la desescalada:** La NN predice en escala `log1p`. Para obtener toneladas reales se aplica `expm1`. Sin embargo, `expm1` puede generar valores extremos si la NN predice valores de log ligeramente fuera del rango de entrenamiento en los outliers. Para evitar que esto destruya el R², se capea la prediccion log al maximo observado en el dataset antes de aplicar `expm1`.

In [ ]:
def predecir_nn_m1(model, X_num, X_dep, X_grp, X_ciclo):
    return model.predict(
        {'numericas':    X_num,
         'departamento': X_dep.reshape(-1, 1),
         'grupo_cultivo':X_grp.reshape(-1, 1),
         'ciclo_cultivo':X_ciclo.reshape(-1, 1)},
        verbose=0
    ).flatten()

y_pred_nn_m1_train_log = np.clip(
    predecir_nn_m1(model_nn_m1, X_num_train_m1, X_dep_train_m1, X_grp_train_m1, X_ciclo_train_m1),
    0, LOG_MAX_M1
)
y_pred_nn_m1_test_log = np.clip(
    predecir_nn_m1(model_nn_m1, X_num_test_m1, X_dep_test_m1, X_grp_test_m1, X_ciclo_test_m1),
    0, LOG_MAX_M1
)

r2_nn_m1_train   = r2_score(y_train_nn_m1, y_pred_nn_m1_train_log)
r2_nn_m1_test    = r2_score(y_test_nn_m1,  y_pred_nn_m1_test_log)
rmse_nn_m1_train = np.sqrt(mean_squared_error(y_train_nn_m1, y_pred_nn_m1_train_log))
rmse_nn_m1_test  = np.sqrt(mean_squared_error(y_test_nn_m1,  y_pred_nn_m1_test_log))
mae_nn_m1_test   = mean_absolute_error(y_test_nn_m1, y_pred_nn_m1_test_log)
gap_nn_m1        = r2_nn_m1_train - r2_nn_m1_test

y_pred_nn_m1_test_ton  = np.expm1(y_pred_nn_m1_test_log)
y_pred_nn_m1_train_ton = np.expm1(y_pred_nn_m1_train_log)
y_test_nn_m1_ton       = np.expm1(y_test_nn_m1)
y_train_nn_m1_ton      = np.expm1(y_train_nn_m1)

r2_nn_m1_test_ton   = r2_score(y_test_nn_m1_ton, y_pred_nn_m1_test_ton)
rmse_nn_m1_test_ton = np.sqrt(mean_squared_error(y_test_nn_m1_ton, y_pred_nn_m1_test_ton))

print('=== Red Neuronal con Embeddings M1 ===')
print(f'  R² log entrenamiento: {r2_nn_m1_train:.4f}')
print(f'  R² log prueba:        {r2_nn_m1_test:.4f}')
print(f'  RMSE log prueba:      {rmse_nn_m1_test:.4f}')
print(f'  Gap train/test:       {gap_nn_m1:.4f}{"  overfitting" if gap_nn_m1 > 0.05 else "  generaliza bien"}')
print(f'  R² toneladas prueba:  {r2_nn_m1_test_ton:.4f}')
print(f'  RMSE toneladas:       {rmse_nn_m1_test_ton:,.0f} t')

#### Visualizacion del desempeno de la Red Neuronal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Actual vs Predicho en toneladas
axes[0].scatter(y_test_nn_m1_ton, y_pred_nn_m1_test_ton, alpha=0.3, color='darkorange', s=10)
lim = max(y_test_nn_m1_ton.max(), y_pred_nn_m1_test_ton.max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Producción Real (t)')
axes[0].set_ylabel('Producción Predicha (t)')
axes[0].set_title(f'NN: Actual vs Predicho (R²={r2_nn_m1_test_ton:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Residuos en escala log
residuos_nn_m1 = y_test_nn_m1 - y_pred_nn_m1_test_log
axes[1].scatter(y_pred_nn_m1_test_log, residuos_nn_m1, alpha=0.3, color='darkorange', s=10)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Producción Predicha (log)')
axes[1].set_ylabel('Residuos (Real − Predicho) en log')
axes[1].set_title('NN: Gráfico de Residuos (escala log)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Desempeño Red Neuronal con Embeddings M1 — Conjunto de Prueba', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Tabla Comparativa Final — ML vs DL

Comparativa directa bajo las mismas condiciones: split 80/20, `random_state=42`, misma variable objetivo `produccion_t`.

In [ ]:
print('=' * 70)
print('  COMPARATIVA FINAL — MÓDULO 1: REGRESIÓN DE PRODUCCIÓN')
print('=' * 70)
print(f'  Target: log1p(produccion_t) — distribución log-normal')
print(f'  Métrica principal: R² en escala log')
print(f'  Features: area_sembrada_ha + es_semestral + OHE(dep+grupo+ciclo)')
print()
print(f'  {"":25} {"Random Forest":>18} {"Red Neuronal":>15}')
print('  ' + '-' * 60)
print(f'  {"R² Log Entrenamiento":25} {r2_rf_m1_train:>18.4f} {r2_nn_m1_train:>15.4f}')
print(f'  {"R² Log Prueba":25} {r2_rf_m1_test:>18.4f} {r2_nn_m1_test:>15.4f}')
print(f'  {"RMSE Log Prueba":25} {rmse_rf_m1_test:>18.4f} {rmse_nn_m1_test:>15.4f}')
print(f'  {"MAE Log Prueba":25} {mae_rf_m1_test:>18.4f} {mae_nn_m1_test:>15.4f}')
print(f'  {"Gap (train−test)":25} {gap_rf_m1:>18.4f} {gap_nn_m1:>15.4f}')
print('  ' + '-' * 60)
print()
print(f'  Referencia en toneladas (para la app):')
print(f'    RF  RMSE: {rmse_rf_m1_test_ton:>12,.0f} t')
print(f'    NN  RMSE: {rmse_nn_m1_test_ton:>12,.0f} t')
print()
ganador_r2  = 'Random Forest' if r2_rf_m1_test  >= r2_nn_m1_test  else 'Red Neuronal'
ganador_gap = 'Random Forest' if abs(gap_rf_m1)  <= abs(gap_nn_m1) else 'Red Neuronal'
print(f'  Mejor R² log:       {ganador_r2}')
print(f'  Menor overfitting:  {ganador_gap}')
print()
print('  Niveles interpretativos para la app:')
for niv, info in NIVELES_PRODUCCION.items():
    print(f'    Nivel {niv} — {info["nombre"]:20} {info["rango"]}')

---
## 11. Conclusiones del Módulo 1

1. **Random Forest** con OHE demostró alta precisión en la predicción de producción agrícola en escala logarítmica.
2. **Red Neuronal con Embeddings** aprende representaciones latentes de departamentos y grupos de cultivo, capturando patrones geográficos y agronómicos.
3. **Target log1p:** La distribución log-normal de la producción justifica el uso de escala logarítmica como métrica principal. El R² en escala original es bajo por los outliers extremos (caña azucarera en el Valle del Cauca).
4. **Features pre-siembra:** Al eliminar `rendimiento_t_ha` (dato post-cosecha), el modelo predice con información genuinamente disponible antes de sembrar.
5. **La app** convierte las predicciones log con `expm1` y las clasifica en 5 niveles interpretativos (Subsistencia → Agroindustrial).


In [ ]:
# ─────────────────────────────────────────────────────────────
# Renombrar objetos M1 con sufijo _m1 para evitar colisiones con M2
# ─────────────────────────────────────────────────────────────
# rf_opt_m1 ya está definido en la celda de GridSearch
# model_nn_m1 ya está definido en la arquitectura NN

# Objetos de preprocesamiento
scaler_num_m1_rf = scaler_num_m1   # StandardScaler de features numéricas (para RF)
ohe_m1_rf        = ohe_m1          # OneHotEncoder (para RF)
scaler_nn_m1_enc = scaler_nn_m1    # StandardScaler de features NN
log_max_m1       = LOG_MAX_M1
niveles_m1       = NIVELES_PRODUCCION
fn_nivel_m1      = clasificar_nivel_produccion

print('Objetos M1 disponibles:')
print(f'  rf_opt_m1:        {type(rf_opt_m1).__name__}')
print(f'  model_nn_m1:      {model_nn_m1.name}')
print(f'  scaler_num_m1_rf: {scaler_num_m1_rf.__class__.__name__} ({scaler_num_m1_rf.n_features_in_} features)')
print(f'  ohe_m1_rf:        {ohe_m1_rf.__class__.__name__} ({len(ohe_m1_rf.get_feature_names_out())} columnas OHE)')
print(f'  scaler_nn_m1_enc: {scaler_nn_m1_enc.__class__.__name__} ({scaler_nn_m1_enc.n_features_in_} features)')
print(f'  log_max_m1:       {log_max_m1:.3f} = {np.expm1(log_max_m1):,.0f} t máximo')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Guardar modelos M1 en Google Drive
# Persistencia entre sesiones — cargar en Sección 6 el día
# de la presentación sin necesidad de reentrenar
# ─────────────────────────────────────────────────────────────
joblib.dump(rf_opt_m1,        f'{RUTA_MODELOS}/m1_rf_opt.pkl')
joblib.dump(scaler_num_m1_rf, f'{RUTA_MODELOS}/m1_scaler_num_rf.pkl')
joblib.dump(ohe_m1_rf,        f'{RUTA_MODELOS}/m1_ohe_rf.pkl')
joblib.dump(scaler_nn_m1_enc, f'{RUTA_MODELOS}/m1_scaler_num_nn.pkl')
joblib.dump(le_dep_m1,        f'{RUTA_MODELOS}/m1_le_dep.pkl')
joblib.dump(le_grp_m1,        f'{RUTA_MODELOS}/m1_le_grp.pkl')
joblib.dump(le_ciclo_m1,      f'{RUTA_MODELOS}/m1_le_ciclo.pkl')
joblib.dump(log_max_m1,       f'{RUTA_MODELOS}/m1_log_max.pkl')
joblib.dump(niveles_m1,       f'{RUTA_MODELOS}/m1_niveles.pkl')
model_nn_m1.save(f'{RUTA_MODELOS}/m1_nn_model.keras')

print('Modelos M1 guardados en Google Drive:')
for f in sorted(os.listdir(RUTA_DRIVE)):
    if f.startswith('m1_'):
        size = os.path.getsize(f'{RUTA_DRIVE}/{f}')
        print(f'  {f:<35} {size/1024:.1f} KB')

---
## Sección 3 — Módulo 2: Clasificación de Producción Alta vs Baja

**Objetivo:** Clasificar si la producción superará la mediana nacional (186 t).
Responde: *¿Vale la pena sembrar este cultivo en esta zona?*

**Comparativa ML vs DL:**
| Modelo | Tipo | Estrategia |
|---|---|---|
| Random Forest Classifier | ML | Ensemble de árboles sobre OHE (50 features) |
| Red Neuronal con Embeddings | DL | Embeddings + sigmoid para clasificación binaria |

**Target:** `prod_alta` = 1 si `produccion_t` > mediana nacional, 0 si no.
Balance garantizado: 50% cada clase por definición matemática de la mediana.


---
## Módulo 2 — Modelo ML: Random Forest Classifier

Se usa **RandomForestClassifier** porque:
- Maneja bien las 50 features OHE sin necesidad de PCA ni escala especial
- Proporciona probabilidades calibradas con `predict_proba`
- Robusto a multicolinealidad — ideal para variables binarias OHE
- Mismo paradigma que M1 (RF Regressor) para coherencia metodológica


## Definición del target — Producción Alta vs Baja

**Target:** `prod_alta` = 1 si `produccion_t` > mediana nacional, 0 si no. Balance perfecto 1:1 por definición matemática.

In [ ]:
# Definir target binario
MEDIANA_PRODUCCION = data_limpia['produccion_t'].median()
data_limpia['prod_alta'] = (data_limpia['produccion_t'] > MEDIANA_PRODUCCION).astype(int)

print(f'Umbral (mediana nacional): {MEDIANA_PRODUCCION:.1f} t')
dist = data_limpia['prod_alta'].value_counts()
for cls, cnt in dist.items():
    label = 'ALTA' if cls == 1 else 'BAJA'
    print(f'  Clase {cls} ({label}): {cnt:,} ({cnt/len(data_limpia)*100:.1f}%)')
print(f'Ratio: {dist.max()/dist.min():.2f}x — perfectamente balanceado')


## Preparación de features M2

Misma estrategia que M1 — OHE directo sin PCA.
Features: `area_sembrada_ha`, `es_semestral` + OHE(`departamento`, `grupo_cultivo`, `ciclo_cultivo`)


In [ ]:
# ─────────────────────────────────────────────────────────────
# Preparación de features M2 — Clasificación
# Misma estrategia que M1: OHE directo sin PCA
# ─────────────────────────────────────────────────────────────
FEATURES_NUM_M2 = ['area_sembrada_ha', 'es_semestral']
FEATURES_CAT_M2 = ['departamento', 'grupo_cultivo', 'ciclo_cultivo']

df_m2 = data_limpia[FEATURES_NUM_M2 + FEATURES_CAT_M2 + ['prod_alta']].copy().reset_index(drop=True)

# One-Hot Encoding
ohe_m2   = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_m2 = ohe_m2.fit_transform(df_m2[FEATURES_CAT_M2])

# Escalado numéricas
scaler_num_m2 = StandardScaler()
X_num_m2      = scaler_num_m2.fit_transform(df_m2[FEATURES_NUM_M2])

# Matriz de features y target
X_m2 = np.hstack([X_num_m2, X_cat_m2])
y_m2 = df_m2['prod_alta'].values

print(f'Features numéricas M2:   {FEATURES_NUM_M2}')
print(f'Features categóricas M2: {FEATURES_CAT_M2}')
print(f'Total features:          {X_m2.shape[1]}')
print(f'Balance target:          {y_m2.mean()*100:.1f}% clase 1 (alta)')
print(f'Umbral (mediana):        {MEDIANA_PRODUCCION:.0f} t')

In [ ]:
X_train_m2, X_test_m2, y_train_m2, y_test_m2 = train_test_split(
    X_m2, y_m2, test_size=0.2, random_state=42, stratify=y_m2
)

print(f'Entrenamiento M2: {X_train_m2.shape[0]:,} x {X_train_m2.shape[1]} features')
print(f'Prueba M2:        {X_test_m2.shape[0]:,}')
print(f'Balance train — clase 0: {(y_train_m2==0).sum():,} | clase 1: {(y_train_m2==1).sum():,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Random Forest Classifier base — línea de referencia
# ─────────────────────────────────────────────────────────────
print('Entrenando Random Forest M2 base (sin optimizar)...')
rf_base_m2 = RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=100)
rf_base_m2.fit(X_train_m2, y_train_m2)

y_pred_rf_m2_base = rf_base_m2.predict(X_test_m2)
acc_rf_m2_base    = accuracy_score(y_test_m2, y_pred_rf_m2_base)
auc_rf_m2_base    = roc_auc_score(y_test_m2, rf_base_m2.predict_proba(X_test_m2)[:, 1])

print(f'RF M2 Base — Accuracy: {acc_rf_m2_base:.4f} | AUC: {auc_rf_m2_base:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Optimización RF M2 con GridSearchCV
# Grid reducido — 8 combinaciones × 3 folds = 24 entrenamientos
# scoring: roc_auc — métrica robusta para clasificación binaria
# ─────────────────────────────────────────────────────────────
param_grid_rf_m2 = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 10],
    'min_samples_split': [2, 5],
}

grid_rf_m2 = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid = param_grid_rf_m2,
    cv         = 3,
    n_jobs     = -1,
    verbose    = 1,
    scoring    = 'roc_auc',
)

print('Iniciando GridSearch RF M2...')
print(f'Combinaciones: {len(param_grid_rf_m2["n_estimators"]) * len(param_grid_rf_m2["max_depth"]) * len(param_grid_rf_m2["min_samples_split"])} × 3 folds = 24 entrenamientos')
grid_rf_m2.fit(X_train_m2, y_train_m2)
print(f'Mejores parámetros: {grid_rf_m2.best_params_}')
rf_opt_m2 = grid_rf_m2.best_estimator_

In [ ]:
# Evaluación completa RF M2
y_pred_rf_m2       = rf_opt_m2.predict(X_test_m2)
y_prob_rf_m2       = rf_opt_m2.predict_proba(X_test_m2)[:, 1]
y_pred_rf_m2_train = rf_opt_m2.predict(X_train_m2)
y_prob_rf_m2_train = rf_opt_m2.predict_proba(X_train_m2)[:, 1]

acc_rf_m2_train  = accuracy_score(y_train_m2, y_pred_rf_m2_train)
acc_rf_m2_test   = accuracy_score(y_test_m2,  y_pred_rf_m2)
prec_rf_m2       = precision_score(y_test_m2,  y_pred_rf_m2)
rec_rf_m2        = recall_score(y_test_m2,     y_pred_rf_m2)
f1_rf_m2         = f1_score(y_test_m2,         y_pred_rf_m2)
auc_rf_m2        = roc_auc_score(y_test_m2,    y_prob_rf_m2)
gap_rf_m2        = acc_rf_m2_train - acc_rf_m2_test

print('=== Random Forest M2 Optimizado ===')
print(f'  Accuracy entrenamiento: {acc_rf_m2_train:.4f}')
print(f'  Accuracy prueba:        {acc_rf_m2_test:.4f}')
print(f'  Precisión:              {prec_rf_m2:.4f}')
print(f'  Recall:                 {rec_rf_m2:.4f}')
print(f'  F1-Score:               {f1_rf_m2:.4f}')
print(f'  AUC-ROC:                {auc_rf_m2:.4f}')
print(f'  Gap train/test:         {gap_rf_m2:.4f}{"  ← overfitting" if gap_rf_m2 > 0.05 else "  ← generaliza bien"}')
print()
print(classification_report(y_test_m2, y_pred_rf_m2, target_names=['Baja', 'Alta']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm_rf_m2 = confusion_matrix(y_test_m2, y_pred_rf_m2)
ConfusionMatrixDisplay(cm_rf_m2, display_labels=['Baja', 'Alta']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'RF M2 — Matriz de Confusión\nAccuracy={acc_rf_m2_test:.4f}')

# Curva ROC
fpr_rf_m2, tpr_rf_m2, _ = roc_curve(y_test_m2, y_prob_rf_m2)
axes[1].plot(fpr_rf_m2, tpr_rf_m2, color='steelblue', label=f'RF (AUC={auc_rf_m2:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('RF M2 — Curva ROC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Desempeño Random Forest M2 — Conjunto de Prueba', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Preparación de datos para la Red Neuronal

La NN usa una arquitectura con **Embeddings** para variables categóricas — aprende representaciones densas de `departamento`, `grupo_cultivo` y `ciclo_cultivo`. Las variables numéricas se escalan con StandardScaler.

**¿Por qué la NN no necesita PCA?**
La NN maneja la multicolinealidad internamente a través de sus pesos — puede aprender a ignorar información redundante entre `area_sembrada_ha` y `area_cosechada_ha`. El PCA solo es necesario para modelos lineales como el SVM.

In [ ]:
FEATURES_NUM_NN_M2 = ['area_sembrada_ha', 'es_semestral']
FEATURES_CAT_NN_M2 = ['departamento', 'grupo_cultivo', 'ciclo_cultivo']

df_nn_m2 = data_limpia[FEATURES_NUM_NN_M2 + FEATURES_CAT_NN_M2 + ['prod_alta']].copy().reset_index(drop=True)

le_dep_m2   = LabelEncoder()
le_grp_m2   = LabelEncoder()
le_ciclo_m2 = LabelEncoder()

df_nn_m2['dep_enc']   = le_dep_m2.fit_transform(df_nn_m2['departamento'])
df_nn_m2['grp_enc']   = le_grp_m2.fit_transform(df_nn_m2['grupo_cultivo'])
df_nn_m2['ciclo_enc'] = le_ciclo_m2.fit_transform(df_nn_m2['ciclo_cultivo'])

n_dep_m2   = df_nn_m2['dep_enc'].nunique()
n_grp_m2   = df_nn_m2['grp_enc'].nunique()
n_ciclo_m2 = df_nn_m2['ciclo_enc'].nunique()

scaler_nn_m2 = StandardScaler()
X_num_nn_m2  = scaler_nn_m2.fit_transform(df_nn_m2[FEATURES_NUM_NN_M2])

X_dep_nn_m2   = df_nn_m2['dep_enc'].values
X_grp_nn_m2   = df_nn_m2['grp_enc'].values
X_ciclo_nn_m2 = df_nn_m2['ciclo_enc'].values
y_nn_m2       = df_nn_m2['prod_alta'].values

indices_m2 = np.arange(len(df_nn_m2))
idx_train_m2, idx_test_m2 = train_test_split(
    indices_m2, test_size=0.2, random_state=42, stratify=y_nn_m2
)

X_num_train_m2,   X_num_test_m2   = X_num_nn_m2[idx_train_m2],   X_num_nn_m2[idx_test_m2]
X_dep_train_m2,   X_dep_test_m2   = X_dep_nn_m2[idx_train_m2],   X_dep_nn_m2[idx_test_m2]
X_grp_train_m2,   X_grp_test_m2   = X_grp_nn_m2[idx_train_m2],   X_grp_nn_m2[idx_test_m2]
X_ciclo_train_m2, X_ciclo_test_m2 = X_ciclo_nn_m2[idx_train_m2], X_ciclo_nn_m2[idx_test_m2]
y_train_m2_nn,    y_test_m2_nn    = y_nn_m2[idx_train_m2],        y_nn_m2[idx_test_m2]

print(f'Entrenamiento NN M2: {len(idx_train_m2):,} | Prueba: {len(idx_test_m2):,}')
print(f'Embeddings — dep: {n_dep_m2} | grp: {n_grp_m2} | ciclo: {n_ciclo_m2}')

## Construcción y Entrenamiento de la Red Neuronal

Arquitectura con Embeddings + capas densas con regularización completa (BatchNormalization + Dropout + EarlyStopping). Salida **sigmoid** para clasificación binaria.

In [ ]:
tf.random.set_seed(42)

n_num_features_m2 = X_num_train_m2.shape[1]
dim_dep_m2   = min(50, n_dep_m2   // 2 + 1)
dim_grp_m2   = min(50, n_grp_m2   // 2 + 1)
dim_ciclo_m2 = min(50, n_ciclo_m2 // 2 + 1)

inp_num_m2   = Input(shape=(n_num_features_m2,), name='numericas')
inp_dep_m2   = Input(shape=(1,),                 name='departamento')
inp_grp_m2   = Input(shape=(1,),                 name='grupo_cultivo')
inp_ciclo_m2 = Input(shape=(1,),                 name='ciclo_cultivo')

emb_dep_m2   = Flatten()(Embedding(n_dep_m2+1,   dim_dep_m2,   name='emb_dep_m2')(inp_dep_m2))
emb_grp_m2   = Flatten()(Embedding(n_grp_m2+1,   dim_grp_m2,   name='emb_grp_m2')(inp_grp_m2))
emb_ciclo_m2 = Flatten()(Embedding(n_ciclo_m2+1, dim_ciclo_m2, name='emb_ciclo_m2')(inp_ciclo_m2))

x_m2 = Concatenate()([inp_num_m2, emb_dep_m2, emb_grp_m2, emb_ciclo_m2])
x_m2 = Dense(128, activation='relu')(x_m2)
x_m2 = BatchNormalization()(x_m2)
x_m2 = Dropout(0.3)(x_m2)
x_m2 = Dense(64, activation='relu')(x_m2)
x_m2 = BatchNormalization()(x_m2)
x_m2 = Dropout(0.2)(x_m2)
x_m2 = Dense(32, activation='relu')(x_m2)

salida_m2 = Dense(1, activation='sigmoid', name='prod_alta')(x_m2)

model_nn_m2 = Model(
    inputs  = [inp_num_m2, inp_dep_m2, inp_grp_m2, inp_ciclo_m2],
    outputs = salida_m2,
    name    = 'AgroInsight_M2_NN'
)
model_nn_m2.compile(
    optimizer = 'adam',
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)
model_nn_m2.summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Entrenamiento NN M2 con EarlyStopping + ReduceLROnPlateau
# ─────────────────────────────────────────────────────────────
early_stop_m2 = EarlyStopping(
    monitor             = 'val_loss',
    patience            = 20,
    restore_best_weights = True,
    verbose             = 1,
)

reduce_lr_m2 = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 8,
    min_lr   = 1e-6,
    verbose  = 1,
)

print('Entrenando Red Neuronal M2...')
history_m2 = model_nn_m2.fit(
    x = {'numericas':    X_num_train_m2,
         'departamento': X_dep_train_m2.reshape(-1, 1),
         'grupo_cultivo':X_grp_train_m2.reshape(-1, 1),
         'ciclo_cultivo':X_ciclo_train_m2.reshape(-1, 1)},
    y                = y_train_m2_nn,
    validation_split = 0.15,
    epochs           = 200,
    batch_size       = 512,
    callbacks        = [early_stop_m2, reduce_lr_m2],
    verbose          = 1,
)

print(f'Epochs ejecutados: {len(history_m2.history["loss"])}')
print(f'Mejor val_loss:    {min(history_m2.history["val_loss"]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_m2.history['loss'],     label='Entrenamiento', color='darkorange')
axes[0].plot(history_m2.history['val_loss'], label='Validación',    color='steelblue', linestyle='--')
axes[0].set_title('NN M2 — Curva de Pérdida (Binary Crossentropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_m2.history['accuracy'],     label='Entrenamiento', color='darkorange')
axes[1].plot(history_m2.history['val_accuracy'], label='Validación',    color='steelblue', linestyle='--')
axes[1].set_title('NN M2 — Curva de Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — Red Neuronal M2', fontsize=13)
plt.tight_layout()
plt.show()

## Evaluación del Modelo de Red Neuronal

In [ ]:
def predecir_nn_m2(model, X_num, X_dep, X_grp, X_ciclo, umbral=0.5):
    probs = model.predict(
        {'numericas':    X_num,
         'departamento': X_dep,
         'grupo_cultivo':X_grp,
         'ciclo_cultivo':X_ciclo},
        verbose=0
    ).flatten()
    return (probs >= umbral).astype(int), probs

y_pred_nn_m2_train, y_prob_nn_m2_train = predecir_nn_m2(
    model_nn_m2,
    X_num_train_m2, X_dep_train_m2.reshape(-1,1),
    X_grp_train_m2.reshape(-1,1), X_ciclo_train_m2.reshape(-1,1)
)
y_pred_nn_m2_test, y_prob_nn_m2_test = predecir_nn_m2(
    model_nn_m2,
    X_num_test_m2, X_dep_test_m2.reshape(-1,1),
    X_grp_test_m2.reshape(-1,1), X_ciclo_test_m2.reshape(-1,1)
)

acc_nn_m2_train = accuracy_score(y_train_m2,   y_pred_nn_m2_train)
acc_nn_m2_test  = accuracy_score(y_test_m2,    y_pred_nn_m2_test)
prec_nn_m2      = precision_score(y_test_m2,   y_pred_nn_m2_test)
rec_nn_m2       = recall_score(y_test_m2,      y_pred_nn_m2_test)
f1_nn_m2        = f1_score(y_test_m2,          y_pred_nn_m2_test)
auc_nn_m2       = roc_auc_score(y_test_m2,     y_prob_nn_m2_test)
gap_nn_m2       = acc_nn_m2_train - acc_nn_m2_test

print('=== Red Neuronal M2 ===')
print(f'  Accuracy entrenamiento: {acc_nn_m2_train:.4f}')
print(f'  Accuracy prueba:        {acc_nn_m2_test:.4f}')
print(f'  AUC-ROC:                {auc_nn_m2:.4f}')
print(f'  Gap train/test:         {gap_nn_m2:.4f}')
print()
print(classification_report(y_test_m2, y_pred_nn_m2_test, target_names=['Baja', 'Alta']))

## Visualización del Desempeño del Modelo de Red Neuronal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_nn_m2 = confusion_matrix(y_test_m2, y_pred_nn_m2_test)
ConfusionMatrixDisplay(cm_nn_m2, display_labels=['Baja', 'Alta']).plot(
    ax=axes[0], cmap='Oranges', colorbar=False
)
axes[0].set_title(f'NN M2 — Matriz de Confusion\nAccuracy={acc_nn_m2_test:.4f}')

fpr_rf_m2_roc, tpr_rf_m2_roc, _ = roc_curve(y_test_m2, y_prob_rf_m2)
fpr_nn_m2_roc, tpr_nn_m2_roc, _ = roc_curve(y_test_m2, y_prob_nn_m2_test)

axes[1].plot(fpr_rf_m2_roc, tpr_rf_m2_roc, color='steelblue',
             lw=2, label=f'Random Forest (AUC={auc_rf_m2:.4f})')
axes[1].plot(fpr_nn_m2_roc, tpr_nn_m2_roc, color='darkorange',
             lw=2, label=f'Red Neuronal  (AUC={auc_nn_m2:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('M2 — Curva ROC: RF vs Red Neuronal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Desempeno Red Neuronal M2 — Conjunto de Prueba', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Análisis Comparativo de Métricas (Entrenamiento vs. Prueba)

In [ ]:
gap_rf_m2_val = acc_rf_m2_train  - acc_rf_m2_test
gap_nn_m2_val = acc_nn_m2_train  - acc_nn_m2_test

print('=' * 65)
print('  COMPARATIVA FINAL — MÓDULO 2: CLASIFICACIÓN ALTA vs BAJA')
print('=' * 65)
print(f'  Umbral: {MEDIANA_PRODUCCION:.0f} t (mediana nacional)')
print(f'  Features: area_sembrada_ha + es_semestral + OHE(dep+grupo+ciclo)')
print()
print(f'  {"":30} {"Random Forest":>15} {"Red Neuronal":>15}')
print('  ' + '-' * 62)
print(f'  {"Accuracy Entrenamiento":30} {acc_rf_m2_train:>15.4f} {acc_nn_m2_train:>15.4f}')
print(f'  {"Accuracy Prueba":30} {acc_rf_m2_test:>15.4f} {acc_nn_m2_test:>15.4f}')
print(f'  {"Precisión":30} {prec_rf_m2:>15.4f} {prec_nn_m2:>15.4f}')
print(f'  {"Recall":30} {rec_rf_m2:>15.4f} {rec_nn_m2:>15.4f}')
print(f'  {"F1-Score":30} {f1_rf_m2:>15.4f} {f1_nn_m2:>15.4f}')
print(f'  {"AUC-ROC":30} {auc_rf_m2:>15.4f} {auc_nn_m2:>15.4f}')
print(f'  {"Gap (train−test)":30} {gap_rf_m2_val:>15.4f} {gap_nn_m2_val:>15.4f}')
print('  ' + '-' * 62)
ganador = 'Random Forest' if auc_rf_m2 >= auc_nn_m2 else 'Red Neuronal'
print(f'  Mejor AUC: {ganador}')

---
## Conclusiones del Módulo 2

1. **Target binario** (producción alta vs baja, umbral = mediana nacional) garantiza balance perfecto 50/50 — sin necesidad de técnicas de balanceo como SMOTE.
2. **Random Forest Classifier** maneja eficientemente las 50 features OHE sin PCA, aprovechando la misma arquitectura de features que M1.
3. **Red Neuronal con Embeddings** aprende representaciones latentes de las categorías — captura patrones geográficos y agronómicos que el RF ve como variables binarias independientes.
4. **Complementariedad M1-M2:** M1 responde *¿cuánto producirá?* y M2 responde *¿vale la pena sembrar?*. Juntos dan al productor una visión completa de la decisión de siembra.
5. **Umbral de decisión:** 186 t — producción mediana nacional 2006-2018.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Renombrar objetos M2 — confirmación de aliases para Drive
# ─────────────────────────────────────────────────────────────
# rf_opt_m2, model_nn_m2 ya tienen sufijo _m2
scaler_num_m2_rf = scaler_num_m2    # StandardScaler numéricas RF M2
ohe_m2_rf        = ohe_m2           # OneHotEncoder M2
scaler_nn_m2_enc = scaler_nn_m2     # StandardScaler numéricas NN M2
# le_dep_m2, le_grp_m2, le_ciclo_m2 ya definidos con sufijo _m2
mediana_m2       = MEDIANA_PRODUCCION

print('Objetos M2 listos:')
print(f'  rf_opt_m2:        {type(rf_opt_m2).__name__}')
print(f'  model_nn_m2:      {model_nn_m2.name}')
print(f'  scaler_num_m2_rf: {scaler_num_m2_rf.__class__.__name__} ({scaler_num_m2_rf.n_features_in_} features)')
print(f'  ohe_m2_rf:        {ohe_m2_rf.__class__.__name__} ({len(ohe_m2_rf.get_feature_names_out())} cols OHE)')
print(f'  scaler_nn_m2_enc: {scaler_nn_m2_enc.__class__.__name__} ({scaler_nn_m2_enc.n_features_in_} features)')
print(f'  mediana_m2:       {mediana_m2:.0f} t')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Guardar modelos M2 en Google Drive
# ─────────────────────────────────────────────────────────────
joblib.dump(rf_opt_m2,        f'{RUTA_MODELOS}/m2_rf_opt.pkl')
joblib.dump(scaler_num_m2_rf, f'{RUTA_MODELOS}/m2_scaler_num_rf.pkl')
joblib.dump(ohe_m2_rf,        f'{RUTA_MODELOS}/m2_ohe_rf.pkl')
joblib.dump(scaler_nn_m2_enc, f'{RUTA_MODELOS}/m2_scaler_num_nn.pkl')
joblib.dump(le_dep_m2,        f'{RUTA_MODELOS}/m2_le_dep.pkl')
joblib.dump(le_grp_m2,        f'{RUTA_MODELOS}/m2_le_grp.pkl')
joblib.dump(le_ciclo_m2,      f'{RUTA_MODELOS}/m2_le_ciclo.pkl')
joblib.dump(mediana_m2,       f'{RUTA_MODELOS}/m2_mediana.pkl')
model_nn_m2.save(f'{RUTA_MODELOS}/m2_nn_model.keras')

print('Modelos M2 guardados en Google Drive:')
for f in sorted(os.listdir(RUTA_DRIVE)):
    if f.startswith('m2_'):
        size = os.path.getsize(f'{RUTA_DRIVE}/{f}')
        print(f'  {f:<35} {size/1024:.1f} KB')

---
## Sección 4 — Módulo 3: Clustering de Municipios

**Objetivo:** Segmentar 1,018 municipios por perfil agrícola histórico.
**ML:** K-Means | **DL:** Autoencoder + Mapas Plotly interactivos

## Preparación del vector municipal

M3 opera a nivel de municipio — se agrega la información histórica de cada municipio en un vector de 5 features con `log1p` para variables con escala muy sesgada.

In [ ]:
# Agregar por municipio — vector histórico 2006-2018
mun = data_limpia.groupby('municipio').agg(
    produccion_total = ('produccion_t',      'sum'),
    rendimiento_prom = ('rendimiento_t_ha',  'mean'),
    area_prom        = ('area_sembrada_ha',  'mean'),
    n_cultivos       = ('cultivo',           'nunique'),
    n_grupos         = ('grupo_cultivo',     'nunique'),
).reset_index()

# Conservar departamento para visualizacion
dep_map = data_limpia.groupby('municipio')['departamento'].first().reset_index()
mun = mun.merge(dep_map, on='municipio')

# log1p para variables con distribucion muy sesgada — igual que M1 con produccion_t
mun['log_produccion'] = np.log1p(mun['produccion_total'])
mun['log_area']       = np.log1p(mun['area_prom'])

FEATURES = ['log_produccion', 'rendimiento_prom', 'log_area', 'n_cultivos', 'n_grupos']

# Escalar — StandardScaler igual que M1 y M2
scaler_m3_raw = StandardScaler()
X_scaled = scaler_m3_raw.fit_transform(mun[FEATURES])

print(f'Vector municipal: {mun.shape[0]} municipios x {len(FEATURES)} features')
print(f'Media por feature (~0): {X_scaled.mean(axis=0).round(3)}')
print(f'Std por feature (~1):   {X_scaled.std(axis=0).round(3)}')

In [ ]:
# Matriz de correlacion del vector municipal
corr_mun = mun[FEATURES].corr()

print('CORRELACIONES ENTRE FEATURES MUNICIPALES:')
print(corr_mun.round(3).to_string())

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_mun, dtype=bool))
sns.heatmap(corr_mun, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Correlaciones — Vector Municipal M3', pad=15)
plt.tight_layout()
plt.show()

print('\nNota: correlacion n_cultivos <-> n_grupos = 0.668 — moderada.')
print('No tan severa como area_sembrada/cosechada (0.977) del M1.')
print('No se aplica PCA — las 5 features aportan informacion diferente.')

---
## Modelo ML — K-Means

Método de referencia para clustering. Usamos el **Elbow Method** y el **Silhouette Score** para determinar el número óptimo de clusters k.

In [ ]:
# Elbow method — probar k de 2 a 8
inertias    = []
silhouettes = []
K_RANGE     = range(2, 9)

print('Evaluando k de 2 a 8...')
print(f'\n{"k":>4} {"Inertia":>12} {"Silhouette":>12}')
print('-' * 30)

for k in K_RANGE:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f'{k:>4} {km.inertia_:>12.0f} {sil:>12.4f}')

K_OPT = list(K_RANGE)[silhouettes.index(max(silhouettes))]
print(f'\nK optimo por Silhouette Score: {K_OPT} (score={max(silhouettes):.4f})')

In [ ]:
# Visualizar Elbow y Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_RANGE), inertias, marker='o', linestyle='--', color='steelblue')
axes[0].set_title('Elbow Method — Inercia por k')
axes[0].set_xlabel('Numero de Clusters (k)')
axes[0].set_ylabel('Inercia (Within-cluster sum of squares)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), silhouettes, marker='o', linestyle='--', color='darkorange')
axes[1].axvline(x=K_OPT, color='red', linestyle=':', label=f'K optimo = {K_OPT}')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xlabel('Numero de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Determinacion del numero optimo de clusters — K-Means', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar K-Means con k optimo
km_final = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
mun['cluster_kmeans'] = km_final.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, mun['cluster_kmeans'])

print(f'=== K-Means con k={K_OPT} ===')
print(f'Silhouette Score: {sil_final:.4f}')
print(f'\nDistribucion por cluster:')
print(mun['cluster_kmeans'].value_counts().sort_index().to_string())

print(f'\nPerfil por cluster (medianas):')
perfil_km = mun.groupby('cluster_kmeans').agg(
    n_municipios     = ('municipio',        'count'),
    prod_mediana_t   = ('produccion_total', 'median'),
    rend_mediana     = ('rendimiento_prom', 'median'),
    area_mediana_ha  = ('area_prom',        'median'),
    n_cultivos_med   = ('n_cultivos',       'median'),
).round(1)
print(perfil_km.to_string())

In [ ]:
# Interpretacion y nombre de cada cluster
print('=== INTERPRETACION DE CLUSTERS ===')
print()
nombres_cluster = {}
for c in sorted(mun['cluster_kmeans'].unique()):
    sub = mun[mun['cluster_kmeans'] == c]
    prod_med  = sub['produccion_total'].median()
    rend_med  = sub['rendimiento_prom'].median()
    n_cult    = sub['n_cultivos'].median()
    top_deps  = sub['departamento'].value_counts().head(3).index.tolist()

    # Asignar nombre interpretable segun perfil
    if prod_med > 500000 and rend_med > 10:
        nombre = 'Alta produccion y rendimiento (agroindustrial)'
    elif prod_med > 200000 and n_cult > 20:
        nombre = 'Alta produccion y diversidad'
    elif n_cult > 18 and prod_med < 200000:
        nombre = 'Diversificado de mediana escala'
    elif prod_med > 100000:
        nombre = 'Produccion media extensiva'
    else:
        nombre = 'Pequena escala o subsistencia'

    nombres_cluster[c] = nombre
    print(f'Cluster {c} — {nombre}')
    print(f'  Municipios: {len(sub)}')
    print(f'  Produccion mediana: {prod_med:,.0f} t')
    print(f'  Rendimiento mediano: {rend_med:.1f} t/ha')
    print(f'  Cultivos distintos: {n_cult:.0f}')
    print(f'  Top departamentos: {top_deps}')
    print()

In [ ]:
# Visualizacion PCA 2D del clustering K-Means

pca_viz = PCA(n_components=2, random_state=42)
X_2d    = pca_viz.fit_transform(X_scaled)

var_exp = pca_viz.explained_variance_ratio_.cumsum()[1] * 100

plt.figure(figsize=(10, 7))
colores = ['#1D9E75', '#E24B4A', '#378ADD', '#EF9F27', '#7F77DD', '#D85A30']

for c in sorted(mun['cluster_kmeans'].unique()):
    mask = mun['cluster_kmeans'] == c
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=colores[c], label=f'Cluster {c} ({mask.sum()} mun.)',
                alpha=0.7, s=40)

plt.title(f'K-Means k={K_OPT} — Espacio PCA 2D\n(varianza explicada: {var_exp:.1f}%)', fontsize=13)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Modelo DL — Autoencoder

El Autoencoder aprende una representación comprimida (espacio latente) del vector municipal. Esta representación captura relaciones no lineales entre las features que K-Means no puede explotar. Luego se aplica K-Means sobre el espacio latente y se compara el Silhouette Score con el K-Means directo.

**Arquitectura:**
- **Encoder**: 5 features → 16 → 8 → 4 (espacio latente)
- **Decoder**: 4 → 8 → 16 → 5 features reconstruidas
- **Loss**: MSE de reconstrucción — el modelo aprende a comprimir y reconstruir el perfil municipal

In [ ]:
tf.random.set_seed(42)

# Split para validacion del autoencoder
X_train_ae, X_val_ae = train_test_split(X_scaled, test_size=0.15, random_state=42)

print(f'Entrenamiento autoencoder: {X_train_ae.shape[0]}')
print(f'Validacion:                {X_val_ae.shape[0]}')

# Arquitectura Autoencoder
DIM_INPUT  = X_scaled.shape[1]  # 5 features
DIM_LATENT = 2                   # espacio latente 2D para visualizacion directa

# Encoder
inp = Input(shape=(DIM_INPUT,), name='input')
x   = Dense(16, activation='relu')(inp)
x   = BatchNormalization()(x)
x   = Dropout(0.2)(x)
x   = Dense(8,  activation='relu')(x)
latente = Dense(DIM_LATENT, activation='linear', name='latente')(x)

# Decoder
x   = Dense(8,  activation='relu')(latente)
x   = Dense(16, activation='relu')(x)
out = Dense(DIM_INPUT, activation='linear', name='reconstruccion')(x)

autoencoder = Model(inputs=inp, outputs=out)
encoder     = Model(inputs=inp, outputs=latente)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Entrenamiento Autoencoder M3 con EarlyStopping + ReduceLROnPlateau
# batch_size=64 para M3 (dataset más pequeño: ~1,018 municipios)
# ─────────────────────────────────────────────────────────────
early_stop_ae = EarlyStopping(
    monitor             = 'val_loss',
    patience            = 20,
    restore_best_weights = True,
    verbose             = 1,
)

reduce_lr_ae = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 8,
    min_lr   = 1e-6,
    verbose  = 1,
)

print('Entrenando Autoencoder M3...')
history_ae = autoencoder.fit(
    X_train_ae, X_train_ae,
    validation_data = (X_val_ae, X_val_ae),
    epochs          = 300,
    batch_size      = 64,
    callbacks       = [early_stop_ae, reduce_lr_ae],
    verbose         = 1,
)

print(f'Epochs ejecutados: {len(history_ae.history["loss"])}')
print(f'Loss final:        {history_ae.history["loss"][-1]:.6f}')
print(f'Val loss final:    {history_ae.history["val_loss"][-1]:.6f}')

In [ ]:
# Curva de aprendizaje del Autoencoder
plt.figure(figsize=(10, 5))
plt.plot(history_ae.history['loss'],     label='Entrenamiento', color='steelblue')
plt.plot(history_ae.history['val_loss'], label='Validacion',    color='orange')
plt.title('Curva de Perdida — Autoencoder (MSE Reconstruccion)')
plt.xlabel('Epoca')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Error de reconstruccion por municipio
X_reconstructed = autoencoder.predict(X_scaled, verbose=0)
mun['error_reconstruccion'] = np.mean((X_scaled - X_reconstructed) ** 2, axis=1)

print('Error de reconstruccion por municipio:')
print(f'  Media:  {mun["error_reconstruccion"].mean():.6f}')
print(f'  Mediana:{mun["error_reconstruccion"].median():.6f}')
print(f'  Max:    {mun["error_reconstruccion"].max():.6f}')

print(f'\nMunicipios mas dificiles de reconstruir (posibles anomalias):')
print(mun.nlargest(5, 'error_reconstruccion')[
    ['municipio', 'departamento', 'produccion_total', 'n_cultivos', 'error_reconstruccion']
].to_string())

In [ ]:
# Obtener espacio latente y aplicar K-Means
X_latente = encoder.predict(X_scaled, verbose=0)

# K-Means sobre espacio latente con mismo k optimo
km_ae = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
mun['cluster_ae'] = km_ae.fit_predict(X_latente)

sil_ae = silhouette_score(X_latente, mun['cluster_ae'])
sil_km = silhouette_score(X_scaled,  mun['cluster_kmeans'])

print(f'=== COMPARATIVA DE CALIDAD DE CLUSTERING ===')
print(f'K-Means directo   — Silhouette Score: {sil_km:.4f}')
print(f'Autoencoder + KM  — Silhouette Score: {sil_ae:.4f}')

print(f'\nDistribucion clusters Autoencoder:')
print(mun['cluster_ae'].value_counts().sort_index().to_string())

In [ ]:
# Visualizacion comparativa — K-Means vs Autoencoder en espacio latente
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means en espacio PCA 2D
for c in sorted(mun['cluster_kmeans'].unique()):
    mask = mun['cluster_kmeans'] == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colores[c], label=f'C{c} ({mask.sum()})',
                    alpha=0.7, s=40)
axes[0].set_title(f'K-Means directo (Silhouette={sil_km:.4f})\nEspacio PCA 2D', fontsize=12)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Autoencoder en espacio latente 2D
for c in sorted(mun['cluster_ae'].unique()):
    mask = mun['cluster_ae'] == c
    axes[1].scatter(X_latente[mask, 0], X_latente[mask, 1],
                    c=colores[c], label=f'C{c} ({mask.sum()})',
                    alpha=0.7, s=40)
axes[1].set_title(f'Autoencoder + K-Means (Silhouette={sil_ae:.4f})\nEspacio Latente 2D', fontsize=12)
axes[1].set_xlabel('Dim Latente 1')
axes[1].set_ylabel('Dim Latente 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparativa K-Means vs Autoencoder — M3 Clustering', fontsize=13)
plt.tight_layout()
plt.show()

---
## Análisis Comparativo de Métricas (Entrenamiento vs. Prueba)

In [ ]:
print('=' * 65)
print('  COMPARATIVA FINAL — MODULO 3: CLUSTERING DE MUNICIPIOS')
print('=' * 65)
print(f'  Municipios: {len(mun)}')
print(f'  Features: {FEATURES}')
print(f'  K optimo: {K_OPT} clusters')
print()
print(f'{"":30} {"K-Means":>15} {"Autoencoder":>14}')
print('-' * 60)
print(f'{"Silhouette Score":30} {sil_km:>15.4f} {sil_ae:>14.4f}')
print(f'{"Espacio de clustering":30} {"Original 5D":>15} {"Latente 2D":>14}')
print(f'{"Tipo de separacion":30} {"Lineal":>15} {"No lineal":>14}')
print('-' * 60)

ganador = 'K-Means' if sil_km >= sil_ae else 'Autoencoder'
print(f'\nMejor Silhouette Score: {ganador}')

print(f'\nPerfil de clusters (K-Means):')
print(f'{"Cluster":<10} {"Municipios":<12} {"Prod.Med(t)":<15} {"Rend.Med":<12} {"N.Cult.":<10}')
print('-' * 60)
for c in sorted(mun['cluster_kmeans'].unique()):
    sub = mun[mun['cluster_kmeans']==c]
    print(f'{c:<10} {len(sub):<12} {sub["produccion_total"].median():>14,.0f} {sub["rendimiento_prom"].median():>11.1f} {sub["n_cultivos"].median():>9.0f}')

print(f'\nInterpretacion:')
print(f'  K-Means: rapido e interpretable — separa linealmente en el espacio escalado')
print(f'  Autoencoder: captura relaciones no lineales entre features del perfil municipal')
print(f'  El Silhouette Score mide cohesion intra-cluster y separacion inter-cluster')
print(f'  Valores > 0.5 indican clusters bien definidos')
print(f'  Valores 0.2-0.5 indican estructura debil pero presente — normal en datos geograficos')

## Mapa de Colombia — Clusters por Municipio

El mapa es el output estrella del M3. Muestra visualmente los patrones agrícolas regionales de Colombia coloreados por cluster.

In [ ]:
# Mapa de barras por departamento — no requiere GeoJSON
# Muestra distribucion de clusters por departamento

cluster_dep = mun.groupby(['departamento', 'cluster_kmeans']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 8))
cluster_dep.plot(kind='bar', stacked=True, ax=ax,
                 color=colores[:K_OPT])

ax.set_title('Distribucion de Clusters Agricolas por Departamento', fontsize=14)
ax.set_xlabel('Departamento')
ax.set_ylabel('Numero de Municipios')
ax.legend([f'Cluster {c}' for c in range(K_OPT)],
          title='Cluster', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=90)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in sorted(mun['cluster_kmeans'].unique()):
    mask = mun['cluster_kmeans'] == c
    axes[0].scatter(
        np.log1p(mun.loc[mask, 'produccion_total']),
        mun.loc[mask, 'rendimiento_prom'],
        c=colores[c], label=f'Cluster {c}', alpha=0.7, s=40
    )
axes[0].set_title('K-Means: Produccion vs Rendimiento', fontsize=12)
axes[0].set_xlabel('log(Produccion Total)')
axes[0].set_ylabel('Rendimiento Promedio (t/ha)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for c in sorted(mun['cluster_ae'].unique()):
    mask = mun['cluster_ae'] == c
    axes[1].scatter(
        np.log1p(mun.loc[mask, 'produccion_total']),
        mun.loc[mask, 'rendimiento_prom'],
        c=colores[c], label=f'Cluster {c}', alpha=0.7, s=40
    )
axes[1].set_title('Autoencoder: Produccion vs Rendimiento', fontsize=12)
axes[1].set_xlabel('log(Produccion Total)')
axes[1].set_ylabel('Rendimiento Promedio (t/ha)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Separacion de Clusters — Produccion vs Rendimiento', fontsize=13)
plt.tight_layout()
plt.show()

---
## Conclusiones del Módulo 3

1. **Vector municipal de 5 features** captura el perfil agrícola histórico de cada municipio: producción total, rendimiento, área, diversidad de cultivos y grupos. La transformación `log1p` fue clave para normalizar `produccion_total` (rango de 44 a 47M t).

2. **K-Means** identificó clusters interpretables con patrones geográficos claros. El Silhouette Score mide la cohesión intra-cluster y la separación inter-cluster — valores entre 0.2 y 0.4 son normales en datos geográficos con fronteras difusas.

3. **El Autoencoder** aprende una representación latente no lineal del perfil municipal. Al aplicar K-Means sobre el espacio latente, los clusters capturan relaciones entre features que K-Means directo no puede detectar.

4. **El error de reconstrucción del Autoencoder** identifica municipios con perfiles atípicos difíciles de representar — este output conecta directamente con el M4 de detección de anomalías.

5. **El mapa por departamento** revela la heterogeneidad agrícola regional de Colombia — cada departamento tiene una composición distinta de perfiles municipales que refleja sus condiciones agroecológicas y económicas particulares.

---
**Próximo módulo — M4:** Detección de anomalías en rendimiento (Isolation Forest vs Autoencoder de reconstrucción).

---
## Mapa Interactivo de Colombia — Perfiles Agrícolas por Departamento

### ¿Qué responde este mapa?
*¿En qué zona de Colombia tiene sentido sembrar cierto cultivo según el perfil agrícola histórico del territorio?*

Cada departamento está coloreado según su **perfil agrícola dominante** — el cluster que representa la mayoría de sus municipios.
Al hacer hover sobre un departamento se muestra:
- Perfil dominante y número de municipios
- Producción total histórica 2006-2018
- Rendimiento promedio t/ha
- Diversidad de cultivos
- Composición exacta de clusters dentro del departamento
- **Recomendación de uso agrícola** según el perfil

### Perfiles definidos con k=4
| Cluster | Perfil | Característica principal |
|---|---|---|
| 0 | Pequeña escala / subsistencia | Baja producción, pocos cultivos |
| 1 | Media escala extensiva | Gran área, producción media |
| 2 | Diversificado mediana escala | Alta variedad de cultivos |
| 3 | Alta producción agroindustrial | Máximo rendimiento y producción |

**Instalación requerida:** `!pip install plotly -q`

In [ ]:
# plotly ya instalado en Sección 0
print("plotly disponible OK")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuración k=4 — perfiles asignados dinámicamente
# según producción mediana real de cada cluster
# ─────────────────────────────────────────────────────────────
K_MAPA = 4

km_k4 = KMeans(n_clusters=K_MAPA, random_state=42, n_init=10)
mun['cluster_k4'] = km_k4.fit_predict(X_scaled)

# Ordenar clusters por producción mediana real
resumen = mun.groupby('cluster_k4').agg(
    prod_mediana = ('produccion_total', 'median'),
).sort_values('prod_mediana', ascending=False)
orden = resumen.index.tolist()

print('Clusters ordenados por produccion mediana:')
print(resumen)

# Asignar perfiles según datos reales — no hardcodeado
PERFILES = {
    orden[0]: {'nombre': 'Alta producción agroindustrial',
               'color': '#C0392B',
               'recomendacion': 'Caña azucarera, palma, arroz, cultivos industriales'},
    orden[1]: {'nombre': 'Grande / Alta diversidad',
               'color': '#E67E22',
               'recomendacion': 'Múltiples cultivos viables — zona apta para diversificación'},
    orden[2]: {'nombre': 'Media escala extensiva',
               'color': '#2980B9',
               'recomendacion': 'Cereales, tubérculos, leguminosas a escala comercial'},
    orden[3]: {'nombre': 'Pequeña escala / Subsistencia',
               'color': '#27AE60',
               'recomendacion': 'Frutales menores, hortalizas, cultivos de pancoger'},
}

mun['perfil_nombre'] = mun['cluster_k4'].map(lambda x: PERFILES[x]['nombre'])

print()
vc = mun[mun['departamento']=='VALLE DEL CAUCA']['perfil_nombre'].value_counts().index[0]
print(f'Valle del Cauca — perfil dominante: {vc}')

In [ ]:
# K_MAPA, PERFILES y cluster_k4 ya están definidos en la celda anterior
# Solo importar librerías necesarias para los mapas
print('Librerías para mapas Plotly cargadas OK')
print(f'Usando k={K_MAPA} clusters para los mapas')
print(f'Perfiles definidos: {len(PERFILES)}')

In [ ]:
GEOJSON_URL = 'https://gist.github.com/john-guerra/43c7656821069d00dcbc/raw/be6a6e239cd5b5b803c6e7c2ec405b793a9064dd/Colombia.geo.json'
response = requests.get(GEOJSON_URL, timeout=60)
geojson_colombia = response.json()
print(f'GeoJSON cargado: {len(geojson_colombia["features"])} departamentos')

def normalizar(nombre):
    s = str(nombre).upper().strip()
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

for feature in geojson_colombia['features']:
    nombre = feature['properties'].get('NOMBRE_DPT', '')
    feature['properties']['dep_join'] = normalizar(nombre)

nombres_geo = {f['properties']['dep_join'] for f in geojson_colombia['features']}
print(f'Departamentos GeoJSON normalizados: {len(nombres_geo)}')

In [ ]:
# Construir dataset por departamento
dep_stats = mun.groupby('departamento').agg(
    n_municipios     = ('municipio',        'count'),
    produccion_total = ('produccion_total', 'sum'),
    rendimiento_prom = ('rendimiento_prom', 'mean'),
    n_cultivos_prom  = ('n_cultivos',       'mean'),
).reset_index()

cluster_dom = mun.groupby('departamento')['cluster_k4'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
cluster_dom.columns = ['departamento', 'cluster_dominante']

dep_mapa = dep_stats.merge(cluster_dom, on='departamento')
dep_mapa['dep_join']      = dep_mapa['departamento'].apply(normalizar)
dep_mapa['perfil']        = dep_mapa['cluster_dominante'].map(lambda x: PERFILES[x]['nombre'])
dep_mapa['recomendacion'] = dep_mapa['cluster_dominante'].map(lambda x: PERFILES[x]['recomendacion'])
dep_mapa['prod_fmt']      = dep_mapa['produccion_total'].apply(lambda x: f'{x/1e6:.1f}M t')
dep_mapa['rend_fmt']      = dep_mapa['rendimiento_prom'].apply(lambda x: f'{x:.1f} t/ha')
dep_mapa['cult_fmt']      = dep_mapa['n_cultivos_prom'].apply(lambda x: f'{x:.0f} cultivos/mun.')

correcciones = {
    'BOGOTA':    'SANTAFE DE BOGOTA D.C',
    'BOGOTA D.C':'SANTAFE DE BOGOTA D.C',
    'SAN ANDRES':'ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA',
}
for idx, row in dep_mapa.iterrows():
    if row['dep_join'] not in nombres_geo:
        for k, v in correcciones.items():
            if k in row['dep_join']:
                dep_mapa.at[idx, 'dep_join'] = normalizar(v)

print(f'dep_mapa listo: {len(dep_mapa)} departamentos')
print(dep_mapa[['departamento','perfil']].sort_values('departamento').to_string())

In [ ]:
COLORES_PERFIL = {PERFILES[k]['nombre']: PERFILES[k]['color'] for k in PERFILES}

fig1 = px.choropleth(
    dep_mapa,
    geojson=geojson_colombia,
    locations='dep_join',
    featureidkey='properties.dep_join',
    color='perfil',
    color_discrete_map=COLORES_PERFIL,
    hover_name='departamento',
    hover_data={
        'dep_join': False, 'perfil': True, 'recomendacion': True,
        'n_municipios': True, 'prod_fmt': True, 'rend_fmt': True, 'cult_fmt': True,
    },
    labels={
        'perfil': 'Perfil Dominante', 'recomendacion': 'Cultivos Recomendados',
        'n_municipios': 'Municipios', 'prod_fmt': 'Produccion Total 2006-2018',
        'rend_fmt': 'Rendimiento Promedio', 'cult_fmt': 'Diversidad',
    },
    title='<b>AgroInsight — Perfil agricola dominante por region</b>',
)
fig1.update_geos(fitbounds='locations', visible=False, bgcolor='#D6EAF8')
fig1.update_layout(
    height=750, paper_bgcolor='white',
    legend=dict(title='<b>Perfil Agricola</b>', bgcolor='rgba(255,255,255,0.95)',
                bordercolor='#AAAAAA', borderwidth=1, font=dict(size=12),
                yanchor='top', y=0.98, xanchor='left', x=0.01),
    title=dict(font=dict(size=15), x=0.5, xanchor='center'),
    margin={'r':10,'t':90,'l':10,'b':10},
    annotations=[dict(
        text='Fuente: MADR — EVA V1 2006-2018 · AgroInsight Bootcamp Talento Tech',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.02, xanchor='center', font=dict(size=10, color='gray')
    )]
)
fig1.show()
fig1.write_html(f'{RUTA_MAPAS}/mapa_perfiles_agricolas.html')
print('Mapa 1 guardado OK — Valle del Cauca debería ser ROJO')

In [ ]:
fig2 = px.choropleth(
    dep_mapa,
    geojson=geojson_colombia,
    locations='dep_join',
    featureidkey='properties.dep_join',
    color='rendimiento_prom',
    color_continuous_scale='YlOrRd',
    hover_name='departamento',
    hover_data={
        'dep_join': False, 'perfil': True, 'recomendacion': True,
        'rend_fmt': True, 'prod_fmt': True, 'cult_fmt': True, 'n_municipios': True,
    },
    labels={
        'rendimiento_prom': 'Rendimiento (t/ha)', 'perfil': 'Perfil Dominante',
        'recomendacion': 'Cultivos Recomendados', 'rend_fmt': 'Rendimiento Promedio',
        'prod_fmt': 'Produccion Total 2006-2018', 'n_municipios': 'Municipios',
        'cult_fmt': 'Diversidad',
    },
    title='<b>AgroInsight — Donde es mas productivo el suelo agricola?</b>',
)
fig2.update_geos(fitbounds='locations', visible=False, bgcolor='#D6EAF8')
fig2.update_coloraxes(colorbar=dict(title='Rendimiento<br>(t/ha)', thickness=15, len=0.6))
fig2.update_layout(
    height=750, paper_bgcolor='white',
    title=dict(font=dict(size=15), x=0.5, xanchor='center'),
    margin={'r':10,'t':90,'l':10,'b':10},
    annotations=[dict(
        text='Fuente: MADR — EVA V1 2006-2018 · AgroInsight Bootcamp Talento Tech',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.02, xanchor='center', font=dict(size=10, color='gray')
    )]
)
fig2.show()
fig2.write_html(f'{RUTA_MAPAS}/mapa_rendimiento_colombia.html')
print('Mapa 2 guardado OK')

In [ ]:
dep_mapa['log_produccion_dep'] = np.log1p(dep_mapa['produccion_total'])

fig3 = px.choropleth(
    dep_mapa,
    geojson=geojson_colombia,
    locations='dep_join',
    featureidkey='properties.dep_join',
    color='log_produccion_dep',
    color_continuous_scale='Greens',
    hover_name='departamento',
    hover_data={
        'dep_join': False, 'log_produccion_dep': False,
        'perfil': True, 'recomendacion': True, 'prod_fmt': True,
        'rend_fmt': True, 'n_municipios': True, 'cult_fmt': True,
    },
    labels={
        'log_produccion_dep': 'Log(Produccion)', 'perfil': 'Perfil Dominante',
        'prod_fmt': 'Produccion Total 2006-2018', 'rend_fmt': 'Rendimiento Promedio',
        'n_municipios': 'Municipios', 'cult_fmt': 'Diversidad',
    },
    title='<b>AgroInsight — Cuales son los grandes polos de produccion agricola?</b>',
)
fig3.update_geos(fitbounds='locations', visible=False, bgcolor='#D6EAF8')
fig3.update_coloraxes(colorbar=dict(title='Log(Produccion<br>Total)', thickness=15, len=0.6))
fig3.update_layout(
    height=750, paper_bgcolor='white',
    title=dict(font=dict(size=15), x=0.5, xanchor='center'),
    margin={'r':10,'t':90,'l':10,'b':10},
    annotations=[dict(
        text='Fuente: MADR — EVA V1 2006-2018 · AgroInsight Bootcamp Talento Tech',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.02, xanchor='center', font=dict(size=10, color='gray')
    )]
)
fig3.show()
fig3.write_html(f'{RUTA_MAPAS}/mapa_produccion_colombia.html')
print('Mapa 3 guardado OK')

In [ ]:
import joblib, os, shutil
import numpy as np

# ─────────────────────────────────────────────────────────────
# Guardar modelos M3
# ─────────────────────────────────────────────────────────────
scaler_m3 = scaler_m3_raw

joblib.dump(km_final,  f'{RUTA_MODELOS}/m3_kmeans.pkl')
joblib.dump(scaler_m3, f'{RUTA_MODELOS}/m3_scaler.pkl')
joblib.dump(mun,       f'{RUTA_MODELOS}/m3_mun_df.pkl')
autoencoder.save(f'{RUTA_MODELOS}/m3_autoencoder.keras')
encoder.save(f'{RUTA_MODELOS}/m3_encoder.keras')

# ─────────────────────────────────────────────────────────────
# Guardar X_scaled y FEATURES para no recalcular
# ─────────────────────────────────────────────────────────────
joblib.dump(X_scaled, f'{RUTA_MODELOS}/m3_X_scaled.pkl')
joblib.dump(FEATURES, f'{RUTA_MODELOS}/m3_features.pkl')
joblib.dump(K_OPT,    f'{RUTA_MODELOS}/m3_k_opt.pkl')
joblib.dump(sil_km,   f'{RUTA_MODELOS}/m3_sil_km.pkl')
joblib.dump(sil_ae,   f'{RUTA_MODELOS}/m3_sil_ae.pkl')

print('Modelos M3 guardados OK')

# ─────────────────────────────────────────────────────────────
# Guardar mapas HTML en RUTA_MAPAS
# ─────────────────────────────────────────────────────────────
for fname in ['mapa_perfiles_agricolas.html',
              'mapa_rendimiento_colombia.html',
              'mapa_produccion_colombia.html']:
    origen = f'{RUTA_MAPAS}/{fname}'
    if os.path.exists(origen):
        size = os.path.getsize(origen) / (1024*1024)
        print(f'  [OK] {fname} ({size:.1f} MB)')
    else:
        print(f'  [FALTA] {fname} — correr celdas de mapas primero')

print()
print('M3 completamente guardado en Drive')

### Cómo usar los 3 mapas en la presentación

**Mapa 1 — Perfiles por cluster** → *¿Qué tipo de agricultura domina esta región?*
Útil para un productor o el Ministerio que quiere saber si un departamento es apto para agricultura industrial, diversificada o de subsistencia.

**Mapa 2 — Rendimiento por departamento** → *¿Dónde es más productivo el suelo por hectárea?*
Útil para una aseguradora evaluando riesgo agrícola por zona o un banco definiendo tasas de crédito diferenciadas.

**Mapa 3 — Producción total** → *¿Cuáles son los grandes polos productivos del país?*
Útil para el Ministerio priorizando inversión en infraestructura de transporte y almacenamiento.

---
## Sección 6 — Cargar modelos guardados

### ⚡ DÍA DE LA PRESENTACIÓN — Solo correr esta sección

Carga todos los modelos desde Drive **sin necesidad de reentrenar**.
Requiere haber corrido el notebook completo al menos una vez antes.

**Orden de ejecución el día de la presentación:**
1. Sección 0 — montar Drive
2. Sección de Importaciones globales  
3. **Esta sección (Sección 6)**
4. Sección 7 — exportar para Streamlit (si es necesario actualizar)


In [ ]:
print('Cargando modelos desde Google Drive...')
print(f'  Modelos: {RUTA_MODELOS}')
print()

# ─── M1 ────────────────────────────────────────────────────────
rf_opt_m1        = joblib.load(f'{RUTA_MODELOS}/m1_rf_opt.pkl')
scaler_num_m1_rf = joblib.load(f'{RUTA_MODELOS}/m1_scaler_num_rf.pkl')
ohe_m1_rf        = joblib.load(f'{RUTA_MODELOS}/m1_ohe_rf.pkl')
scaler_nn_m1_enc = joblib.load(f'{RUTA_MODELOS}/m1_scaler_num_nn.pkl')
le_dep_m1        = joblib.load(f'{RUTA_MODELOS}/m1_le_dep.pkl')
le_grp_m1        = joblib.load(f'{RUTA_MODELOS}/m1_le_grp.pkl')
le_ciclo_m1      = joblib.load(f'{RUTA_MODELOS}/m1_le_ciclo.pkl')
log_max_m1       = joblib.load(f'{RUTA_MODELOS}/m1_log_max.pkl')
niveles_m1       = joblib.load(f'{RUTA_MODELOS}/m1_niveles.pkl')
model_nn_m1      = tf.keras.models.load_model(f'{RUTA_MODELOS}/m1_nn_model.keras')
print('  M1 OK')

# ─── M2 ────────────────────────────────────────────────────────
rf_opt_m2        = joblib.load(f'{RUTA_MODELOS}/m2_rf_opt.pkl')
scaler_num_m2_rf = joblib.load(f'{RUTA_MODELOS}/m2_scaler_num_rf.pkl')
ohe_m2_rf        = joblib.load(f'{RUTA_MODELOS}/m2_ohe_rf.pkl')
scaler_nn_m2_enc = joblib.load(f'{RUTA_MODELOS}/m2_scaler_num_nn.pkl')
le_dep_m2        = joblib.load(f'{RUTA_MODELOS}/m2_le_dep.pkl')
le_grp_m2        = joblib.load(f'{RUTA_MODELOS}/m2_le_grp.pkl')
le_ciclo_m2      = joblib.load(f'{RUTA_MODELOS}/m2_le_ciclo.pkl')
mediana_m2       = joblib.load(f'{RUTA_MODELOS}/m2_mediana.pkl')
model_nn_m2      = tf.keras.models.load_model(f'{RUTA_MODELOS}/m2_nn_model.keras')
print('  M2 OK')

# ─── M3 ────────────────────────────────────────────────────────
km_m3     = joblib.load(f'{RUTA_MODELOS}/m3_kmeans.pkl')
scaler_m3 = joblib.load(f'{RUTA_MODELOS}/m3_scaler.pkl')
mun       = joblib.load(f'{RUTA_MODELOS}/m3_mun_df.pkl')
X_scaled  = joblib.load(f'{RUTA_MODELOS}/m3_X_scaled.pkl')
FEATURES  = joblib.load(f'{RUTA_MODELOS}/m3_features.pkl')
K_OPT     = joblib.load(f'{RUTA_MODELOS}/m3_k_opt.pkl')
sil_km    = joblib.load(f'{RUTA_MODELOS}/m3_sil_km.pkl')
sil_ae    = joblib.load(f'{RUTA_MODELOS}/m3_sil_ae.pkl')
print('  M3 OK')

# ─── Datos ─────────────────────────────────────────────────────
data_limpia       = joblib.load(f'{RUTA_MODELOS}/data_limpia.pkl')
listas_categorias = joblib.load(f'{RUTA_MODELOS}/listas_categorias.pkl')
DEPARTAMENTOS     = listas_categorias['departamentos']
GRUPOS_CULTIVO    = listas_categorias['grupos_cultivo']
CICLOS            = listas_categorias['ciclos']

with open(f'{RUTA_MODELOS}/metricas.json', encoding='utf-8') as f_json:
    metricas_guardadas = json.load(f_json)

print(f'  Datos OK — {len(data_limpia):,} registros')
print()

# ─── Constantes ────────────────────────────────────────────────
MEDIANA_PRODUCCION = mediana_m2
LOG_MAX_M1         = log_max_m1
NIVELES_PRODUCCION = niveles_m1

def clasificar_nivel_produccion(toneladas):
    for nivel, info in NIVELES_PRODUCCION.items():
        if toneladas < info['limite']:
            return nivel, f"Nivel {nivel} — {info['nombre']} ({info['rango']})"
    return 5, "Nivel 5 — Agroindustrial (> 1,236 t)"

print('Todos los modelos cargados OK')
print(f'  X_scaled:   {X_scaled.shape}')
print(f'  K_OPT:      {K_OPT}')
print(f'  sil_km:     {sil_km:.4f}')

---
## Sección 7 — Exportar datos para la app Streamlit

Exporta todos los archivos necesarios para correr la app de presentación local.
Esta sección se corre **una sola vez** después del entrenamiento completo.

**Archivos que se exportan a Google Drive:**

| Archivo | Contenido |
|---|---|
| `data_limpia.pkl` | Dataset completo para EDA en Streamlit |
| `metricas.json` | Métricas finales M1, M2 y M3 |
| `listas_categorias.pkl` | Dropdowns de la demo interactiva |
| Modelos M1, M2, M3 | Ya guardados en secciones anteriores |
| Mapas HTML | Ya guardados en sección M3 |


In [ ]:
print('=== Exportando datos para Streamlit ===')
print(f'Destino: {RUTA_DRIVE}')
print()

# ─── 1. Dataset completo para EDA ────────────────────────────
# data_limpia contiene todos los registros limpios con es_semestral
joblib.dump(data_limpia, f'{RUTA_MODELOS}/data_limpia.pkl')
size_dl = os.path.getsize(f'{RUTA_MODELOS}/data_limpia.pkl') / (1024*1024)
print(f'  [OK] data_limpia.pkl           {size_dl:.1f} MB  ({len(data_limpia):,} registros)')

In [ ]:
# ─── 2. Métricas de los modelos en JSON ──────────────────────
# La app Streamlit las usa para mostrar tablas comparativas
# sin necesidad de reentrenar ni cargar los modelos completos
metricas = {
    'M1_Regresion': {
        'target':         'log1p(produccion_t)',
        'features':       'area_sembrada_ha + es_semestral + OHE(dep+grupo+ciclo)',
        'n_features':     int(X_m1.shape[1]),
        'n_train':        int(X_train_m1.shape[0]),
        'n_test':         int(X_test_m1.shape[0]),
        'Random_Forest': {
            'r2_log_train':    round(float(r2_rf_m1_train),   4),
            'r2_log_test':     round(float(r2_rf_m1_test),    4),
            'rmse_log_test':   round(float(rmse_rf_m1_test),  4),
            'mae_log_test':    round(float(mae_rf_m1_test),   4),
            'gap_train_test':  round(float(gap_rf_m1),        4),
            'rmse_ton_test':   round(float(rmse_rf_m1_test_ton), 0),
            'r2_ton_test':     round(float(r2_rf_m1_test_ton),   4),
            'best_params':     grid_rf_m1.best_params_,
        },
        'Red_Neuronal': {
            'r2_log_train':    round(float(r2_nn_m1_train),   4),
            'r2_log_test':     round(float(r2_nn_m1_test),    4),
            'rmse_log_test':   round(float(rmse_nn_m1_test),  4),
            'mae_log_test':    round(float(mae_nn_m1_test),   4),
            'gap_train_test':  round(float(gap_nn_m1),        4),
            'rmse_ton_test':   round(float(rmse_nn_m1_test_ton), 0),
            'r2_ton_test':     round(float(r2_nn_m1_test_ton),   4),
            'arquitectura':    'Embeddings + Dense(128→64→32) + log1p target',
        },
    },
    'M2_Clasificacion': {
        'target':         'prod_alta (0/1)',
        'umbral_t':       round(float(MEDIANA_PRODUCCION), 1),
        'features':       'area_sembrada_ha + es_semestral + OHE(dep+grupo+ciclo)',
        'n_features':     int(X_m2.shape[1]),
        'n_train':        int(X_train_m2.shape[0]),
        'n_test':         int(X_test_m2.shape[0]),
        'Random_Forest': {
            'accuracy_train':  round(float(acc_rf_m2_train), 4),
            'accuracy_test':   round(float(acc_rf_m2_test),  4),
            'precision':       round(float(prec_rf_m2),      4),
            'recall':          round(float(rec_rf_m2),       4),
            'f1_score':        round(float(f1_rf_m2),        4),
            'auc_roc':         round(float(auc_rf_m2),       4),
            'gap_train_test':  round(float(gap_rf_m2),       4),
            'best_params':     grid_rf_m2.best_params_,
        },
        'Red_Neuronal': {
            'accuracy_train':  round(float(acc_nn_m2_train), 4),
            'accuracy_test':   round(float(acc_nn_m2_test),  4),
            'precision':       round(float(prec_nn_m2),      4),
            'recall':          round(float(rec_nn_m2),       4),
            'f1_score':        round(float(f1_nn_m2),        4),
            'auc_roc':         round(float(auc_nn_m2),       4),
            'gap_train_test':  round(float(gap_nn_m2),       4),
            'arquitectura':    'Embeddings + Dense(128→64→32) + sigmoid',
        },
    },
    'M3_Clustering': {
        'n_municipios':   int(len(mun)),
        'features':       ['log_produccion', 'rendimiento_prom', 'log_area', 'n_cultivos', 'n_grupos'],
        'k_optimo':       int(K_OPT),
        'k_mapa':         4,
        'K_Means': {
            'silhouette':      round(float(sil_km), 4),
            'espacio':         'Original 5D',
        },
        'Autoencoder': {
            'silhouette':      round(float(sil_ae), 4),
            'espacio':         'Latente 2D',
            'dim_latente':     2,
            'loss_final':      round(float(history_ae.history['loss'][-1]), 6),
        },
        'perfiles': {
            str(k): {
                'nombre':       v['nombre'],
                'recomendacion':v['recomendacion'],
            } for k, v in PERFILES.items()
        },
    },
    'dataset': {
        'fuente':        'MADR — EVA V1 · datos.gov.co · ID: 2pnw-mmge',
        'periodo':       '2006–2018',
        'registros_raw': 205242,
        'registros_limpios': int(len(data_limpia)),
        'departamentos': int(data_limpia['departamento'].nunique()),
        'municipios':    int(data_limpia['municipio'].nunique()),
        'cultivos':      int(data_limpia['cultivo'].nunique()),
        'grupos':        int(data_limpia['grupo_cultivo'].nunique()),
    }
}

ruta_json = f'{RUTA_MODELOS}/metricas.json'
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

size_j = os.path.getsize(ruta_json) / 1024
print(f'  [OK] metricas.json             {size_j:.1f} KB')
print()
print('  Contenido metricas.json:')
print(f'    M1 RF  — R² log prueba:  {metricas["M1_Regresion"]["Random_Forest"]["r2_log_test"]}')
print(f'    M1 NN  — R² log prueba:  {metricas["M1_Regresion"]["Red_Neuronal"]["r2_log_test"]}')
print(f'    M2 RF  — AUC-ROC:        {metricas["M2_Clasificacion"]["Random_Forest"]["auc_roc"]}')
print(f'    M2 NN  — AUC-ROC:        {metricas["M2_Clasificacion"]["Red_Neuronal"]["auc_roc"]}')
print(f'    M3 KM  — Silhouette:     {metricas["M3_Clustering"]["K_Means"]["silhouette"]}')
print(f'    M3 AE  — Silhouette:     {metricas["M3_Clustering"]["Autoencoder"]["silhouette"]}')

In [ ]:
# ─── 3. Listas de categorías para dropdowns Streamlit ────────
listas_categorias = {
    'departamentos':  sorted(data_limpia['departamento'].unique().tolist()),
    'grupos_cultivo': sorted(data_limpia['grupo_cultivo'].unique().tolist()),
    'ciclos':         sorted(data_limpia['ciclo_cultivo'].unique().tolist()),
}
joblib.dump(listas_categorias, f'{RUTA_MODELOS}/listas_categorias.pkl')
size_lc = os.path.getsize(f'{RUTA_MODELOS}/listas_categorias.pkl') / 1024
print(f'  [OK] listas_categorias.pkl     {size_lc:.1f} KB')
print(f'       dep:{len(listas_categorias["departamentos"])} | grp:{len(listas_categorias["grupos_cultivo"])} | ciclos:{len(listas_categorias["ciclos"])}')

# ─── 4. Exportar gráficas clave como PNG ─────────────────────
import matplotlib
matplotlib.use('Agg')  # backend sin ventana para guardar PNG

os.makedirs(f'{RUTA_DRIVE}/graficas', exist_ok=True)

# Gráfica 1 — Distribución producción log vs original
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(data_limpia['produccion_t'],            bins=80, color='steelblue', alpha=0.7)
axes[0].set_title('Distribución Producción — Escala Original')
axes[0].set_xlabel('Producción (t)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(True, alpha=0.3)
axes[1].hist(np.log1p(data_limpia['produccion_t']), bins=80, color='darkorange', alpha=0.7)
axes[1].set_title('Distribución Producción — Escala Log1p')
axes[1].set_xlabel('log1p(Producción)')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Distribución de Producción Agrícola EVA 2006-2018', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/eda_distribucion_produccion.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/eda_distribucion_produccion.png')

# Gráfica 2 — Top 10 departamentos por producción
top_dep = data_limpia.groupby('departamento')['produccion_t'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 5))
top_dep.plot(kind='bar', ax=ax, color='teal', alpha=0.8)
ax.set_title('Top 10 Departamentos — Producción Total 2006-2018', fontsize=13)
ax.set_xlabel('Departamento')
ax.set_ylabel('Producción Total (t)')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/eda_top_departamentos.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/eda_top_departamentos.png')

# Gráfica 3 — Producción por grupo de cultivo
prod_grupo = data_limpia.groupby('grupo_cultivo')['produccion_t'].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
prod_grupo.plot(kind='barh', ax=ax, color='coral', alpha=0.8)
ax.set_title('Producción Total por Grupo de Cultivo 2006-2018', fontsize=13)
ax.set_xlabel('Producción Total (t)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/eda_produccion_por_grupo.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/eda_produccion_por_grupo.png')

# Gráfica 4 — Distribución es_semestral
fig, ax = plt.subplots(figsize=(7, 5))
counts = data_limpia['es_semestral'].value_counts()
ax.bar(['Anual (0)', 'Semestral (1)'], counts.values, color=['steelblue', 'darkorange'], alpha=0.8)
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=11)
ax.set_title('Distribución de Ciclos — Anual vs Semestral', fontsize=13)
ax.set_ylabel('Número de registros')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/eda_es_semestral.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/eda_es_semestral.png')

In [ ]:
os.makedirs(f'{RUTA_DRIVE}/graficas', exist_ok=True)

# Gráfica 5 — M1: Actual vs Predicho RF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test_rf_m1_ton, y_pred_rf_m1_test_ton, alpha=0.3, color='steelblue', s=8)
lim = max(y_test_rf_m1_ton.max(), y_pred_rf_m1_test_ton.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Predicción perfecta')
axes[0].set_title(f'RF M1: Actual vs Predicho (R²={r2_rf_m1_test_ton:.4f})')
axes[0].set_xlabel('Real (t)'); axes[0].set_ylabel('Predicho (t)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].scatter(y_test_nn_m1_ton, y_pred_nn_m1_test_ton, alpha=0.3, color='darkorange', s=8)
lim2 = max(y_test_nn_m1_ton.max(), y_pred_nn_m1_test_ton.max())
axes[1].plot([0, lim2], [0, lim2], 'r--', lw=1.5, label='Predicción perfecta')
axes[1].set_title(f'NN M1: Actual vs Predicho (R²={r2_nn_m1_test_ton:.4f})')
axes[1].set_xlabel('Real (t)'); axes[1].set_ylabel('Predicho (t)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('M1 — Comparativa Actual vs Predicho: RF vs Red Neuronal', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/m1_actual_vs_predicho.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/m1_actual_vs_predicho.png')

# Gráfica 6 — M1: Curvas de aprendizaje NN
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_m1.history['loss'],     label='Entrenamiento', color='steelblue')
axes[0].plot(history_m1.history['val_loss'], label='Validación',    color='orange', ls='--')
axes[0].set_title('NN M1 — Curva de Pérdida'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history_m1.history['mae'],     label='Entrenamiento', color='steelblue')
axes[1].plot(history_m1.history['val_mae'], label='Validación',    color='orange', ls='--')
axes[1].set_title('NN M1 — Curva MAE'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Curvas de Aprendizaje — Red Neuronal M1', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/m1_curvas_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/m1_curvas_aprendizaje.png')

# Gráfica 7 — M2: ROC comparativa RF vs NN
fig, ax = plt.subplots(figsize=(7, 6))
fpr_rf, tpr_rf, _ = roc_curve(y_test_m2, y_prob_rf_m2)
fpr_nn, tpr_nn, _ = roc_curve(y_test_m2, y_prob_nn_m2_test)
ax.plot(fpr_rf, tpr_rf, color='steelblue',  lw=2, label=f'Random Forest (AUC={auc_rf_m2:.4f})')
ax.plot(fpr_nn, tpr_nn, color='darkorange', lw=2, label=f'Red Neuronal  (AUC={auc_nn_m2:.4f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('M2 — Curva ROC: Random Forest vs Red Neuronal', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/m2_roc_comparativa.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/m2_roc_comparativa.png')

# Gráfica 8 — M2: Curvas de aprendizaje NN
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_m2.history['loss'],     label='Entrenamiento', color='darkorange')
axes[0].plot(history_m2.history['val_loss'], label='Validación',    color='steelblue', ls='--')
axes[0].set_title('NN M2 — Curva de Pérdida'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history_m2.history['accuracy'],     label='Entrenamiento', color='darkorange')
axes[1].plot(history_m2.history['val_accuracy'], label='Validación',    color='steelblue', ls='--')
axes[1].set_title('NN M2 — Curva Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Curvas de Aprendizaje — Red Neuronal M2', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/m2_curvas_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/m2_curvas_aprendizaje.png')

# Gráfica 9 — M3: Scatter clusters
pca_v = PCA_viz(n_components=2, random_state=42)
X_2d_v = pca_v.fit_transform(X_scaled)
colores_m3 = ['#1D9E75','#E24B4A','#378ADD','#EF9F27']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for c in sorted(mun['cluster_kmeans'].unique()):
    mask = mun['cluster_kmeans'] == c
    axes[0].scatter(X_2d_v[mask,0], X_2d_v[mask,1],
                    c=colores_m3[c % 4], label=f'Cluster {c}', alpha=0.7, s=30)
axes[0].set_title(f'K-Means (Silhouette={sil_km:.4f})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
for c in sorted(mun['cluster_ae'].unique()):
    mask = mun['cluster_ae'] == c
    axes[1].scatter(X_latente[mask,0], X_latente[mask,1],
                    c=colores_m3[c % 4], label=f'Cluster {c}', alpha=0.7, s=30)
axes[1].set_title(f'Autoencoder (Silhouette={sil_ae:.4f})')
axes[1].set_xlabel('Dim Latente 1'); axes[1].set_ylabel('Dim Latente 2')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('M3 — Comparativa K-Means vs Autoencoder', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICAS}/m3_clusters_comparativa.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'  [OK] graficas/m3_clusters_comparativa.png')

In [ ]:
# ─── 5. Verificación final de todo lo exportado ───────────────
print('=' * 60)
print('  VERIFICACIÓN FINAL — ARCHIVOS PARA STREAMLIT')
print('=' * 60)
print()

archivos_requeridos = {
    'Modelos M1': [
        f'{RUTA_MODELOS}/m1_rf_opt.pkl',
        f'{RUTA_MODELOS}/m1_scaler_num_rf.pkl',
        f'{RUTA_MODELOS}/m1_ohe_rf.pkl',
        f'{RUTA_MODELOS}/m1_scaler_num_nn.pkl',
        f'{RUTA_MODELOS}/m1_le_dep.pkl',
        f'{RUTA_MODELOS}/m1_le_grp.pkl',
        f'{RUTA_MODELOS}/m1_le_ciclo.pkl',
        f'{RUTA_MODELOS}/m1_log_max.pkl',
        f'{RUTA_MODELOS}/m1_niveles.pkl',
        f'{RUTA_MODELOS}/m1_nn_model.keras',
    ],
    'Modelos M2': [
        f'{RUTA_MODELOS}/m2_rf_opt.pkl',
        f'{RUTA_MODELOS}/m2_scaler_num_rf.pkl',
        f'{RUTA_MODELOS}/m2_ohe_rf.pkl',
        f'{RUTA_MODELOS}/m2_scaler_num_nn.pkl',
        f'{RUTA_MODELOS}/m2_le_dep.pkl',
        f'{RUTA_MODELOS}/m2_le_grp.pkl',
        f'{RUTA_MODELOS}/m2_le_ciclo.pkl',
        f'{RUTA_MODELOS}/m2_mediana.pkl',
        f'{RUTA_MODELOS}/m2_nn_model.keras',
    ],
    'Modelos M3': [
        f'{RUTA_MODELOS}/m3_kmeans.pkl',
        f'{RUTA_MODELOS}/m3_scaler.pkl',
        f'{RUTA_MODELOS}/m3_mun_df.pkl',
        f'{RUTA_MODELOS}/m3_autoencoder.keras',
        f'{RUTA_MODELOS}/m3_encoder.keras',
    ],
    'Datos app': [
        f'{RUTA_MODELOS}/data_limpia.pkl',
        f'{RUTA_MODELOS}/metricas.json',
        f'{RUTA_MODELOS}/listas_categorias.pkl',
    ],
    'Mapas HTML': [
        f'{RUTA_MAPAS}/mapa_perfiles_agricolas.html',
        f'{RUTA_MAPAS}/mapa_rendimiento_colombia.html',
        f'{RUTA_MAPAS}/mapa_produccion_colombia.html',
    ],
    'Graficas PNG': [
        f'{RUTA_GRAFICAS}/eda_distribucion_produccion.png',
        f'{RUTA_GRAFICAS}/eda_top_departamentos.png',
        f'{RUTA_GRAFICAS}/eda_produccion_por_grupo.png',
        f'{RUTA_GRAFICAS}/eda_es_semestral.png',
        f'{RUTA_GRAFICAS}/m1_actual_vs_predicho.png',
        f'{RUTA_GRAFICAS}/m1_curvas_aprendizaje.png',
        f'{RUTA_GRAFICAS}/m2_roc_comparativa.png',
        f'{RUTA_GRAFICAS}/m2_curvas_aprendizaje.png',
        f'{RUTA_GRAFICAS}/m3_clusters_comparativa.png',
    ],
}

total_ok = 0
total_falta = 0
for categoria, archivos in archivos_requeridos.items():
    print(f'  {categoria}:')
    for ruta in archivos:
        nombre = os.path.basename(ruta)
        if os.path.exists(ruta):
            size = os.path.getsize(ruta)
            unit = 'MB' if size > 1024*1024 else 'KB'
            size_fmt = size/(1024*1024) if size > 1024*1024 else size/1024
            print(f'    [OK]    {nombre:<40} {size_fmt:.1f} {unit}')
            total_ok += 1
        else:
            print(f'    [FALTA] {nombre}')
            total_falta += 1
    print()

print('=' * 60)
if total_falta == 0:
    print(f'  LISTO — {total_ok} archivos exportados correctamente')
else:
    print(f'  ATENCIÓN: {total_falta} archivo(s) faltante(s)')
print('=' * 60)